# GRIFFIN Data Pipeline - DHRM Career Group Scraper

**BUAD 5722: Big Data & Cloud Analytics**

GRIFFIN (General Role Information Framework for Finding and Integrating Nomenclature) is the data pipeline
component of the DHRM HR Classification project. Its job is to scrape, parse, and store Virginia Department
of Human Resource Management (DHRM) career group descriptions so they can be used downstream for
classification matching.

## Libraries Used

| Library | Purpose | Course Week |
|---|---|---|
| `requests` | Sends HTTP requests to download DHRM web pages and PDF documents | Week 3, Web APIs |
| `BeautifulSoup` (bs4) | Parses raw HTML into a navigable tree so we can extract structured text | Week 3, Web Scraping |
| `re` | Regular expressions for cleaning whitespace, splitting role blocks, and pattern matching | Week 3, Text Processing |
| `pandas` | Organizes scraped data into DataFrames for inspection, cleaning, and export | Week 2, Tabular Data |
| `sqlalchemy` | Connects to a SQLite database and writes DataFrames as relational tables | Week 5, Databases |
| `pdfplumber` | Opens and extracts text from PDF files (used for the 6 PDF-only career groups) | Week 3, Document Parsing |
| `os` | File and directory management, builds cache paths | Core Python |
| `time` | Adds polite delays between HTTP requests to avoid overwhelming the DHRM server | Core Python |

In [1]:
# ---------------------------------------------------------------------------
# Cell 1 - Setup and Library Imports
# ---------------------------------------------------------------------------

import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from sqlalchemy import create_engine
import pdfplumber
import os
import time

print("All libraries loaded successfully.")

All libraries loaded successfully.


## Define Source URLs

Virginia DHRM organizes state positions into **7 occupational families** containing roughly **56 career groups**.
Each career group page lists the roles (job titles), their role codes, and the knowledge/skill/ability (KSA)
requirements for that group.

- **50 career groups** are published as standard HTML pages with a consistent template.
- **6 career groups** are published only as PDF documents, which require a separate parsing strategy.

All 50 HTML pages share a common base URL on the DHRM itech server, so we store a static
`HTML_BASE_URL` and a dictionary of path suffixes. The full URL for each page is built with
an f-string: `f"{HTML_BASE_URL}{path}"`. The 6 PDF sources use different hosting domains,
so their full URLs are stored directly.

We store every source as a tuple of `(code, name, family, url, format)` so the pipeline can iterate over
them uniformly.

For Phase 1 development, we define a small `TEST_SOURCES` list containing only Financial Services
so we can build and debug the parser before running against all 56 pages.

## Thought Process:

Initially I had Claude generate me some code to execute the work.
However, it had hardcoded all the URLs.

I wanted to follow the in-class principles, and make it easier to maintain, and be more efficient. 
We defined a base URL and then just the unique path suffixes for each career group, going from tactical to programmatic thinking. 
Then we just run the URL as a f-string and do the web scraping.


In [ ]:
# ---------------------------------------------------------------------------
# Cell 2 - Source URL Definitions
# ---------------------------------------------------------------------------

# Static base URL shared by all 50 HTML career group pages
HTML_BASE_URL = "https://web1.dhrm.virginia.gov/itech/DHRMWebAssets/careergroups/"


'''
Initially we hardcoded the full URLs for each HTML page, but that was redundant and error-prone.
I wanted to follow the in-class principle and make it easier to maintain and update the URLs if the structure changes in the future. 
By defining a base URL and then just the unique path suffixes for each career group, we are doing this programmatically. 
This also makes it clearer which parts of the URL are consistent and which parts are specific to each career group.
'''

# I used Claude to generate the list HTML paths from the website and I did some spot checks to verify along with validating the correct number of entries (50). As you'll see later we do have some edge cases. 

# Dynamic path suffixes -- only the parts that change per career group
# Each entry: code -> (name, family, path_suffix)
HTML_PATHS = {
    "19010": ("Administration and Office Support", "Administrative Services", "admin/AdminOfficeSupport19010.htm"),
    "39050": ("Architecture and Engineering Services", "Engineering and Technology", "engtechnology/eng39050ArchEngineer.htm"),
    "19190": ("Audit and Management Services", "Administrative Services", "admin/AuditMgmt19190.htm"),
    "79030": ("Building Trades", "Trades and Operations", "trades/BuildingTrades79030.htm"),
    "39010": ("Computer Operations", "Engineering and Technology", "engtechnology/ComputerOperations39010.htm"),
    "49010": ("Counseling Services", "Health and Human Services", "health/hea49010Counseling.htm"),
    "49030": ("Dental Services", "Health and Human Services", "health/Dental49030.htm"),
    "49050": ("Direct Service", "Health and Human Services", "health/DirectSvcs49050.htm"),
    "29130": ("Education Administration", "Educational and Media Services", "EducMediaServ/edu29130EducAdm.htm"),
    "29140": ("Education Support Services", "Educational and Media Services", "EducMediaServ/edu29140EducSupportServ.htm"),
    "39030": ("Electronics", "Engineering and Technology", "engtechnology/eng39030Electronics.htm"),
    "39070": ("Engineering Technology", "Engineering and Technology", "engtechnology/eng39070EnginTech.htm"),
    "59030": ("Environmental Services", "Natural Resources and Applied Science", "natresouc/nat59030Environmental.htm"),
    "79050": ("Equipment Service and Repair", "Trades and Operations", "trades/EquipSvcsRepair79050.htm"),
    "19030": ("Financial Services", "Administrative Services", "admin/FinancialServices19030.htm"),
    "79210": ("Food Services", "Trades and Operations", "trades/FoodService79210.htm"),
    "69130": ("Forensic Science", "Public Safety", "pubsafe/ForensicScience69130.htm"),
    "19220": ("General Administration", "Administrative Services", "admin/GenlAdmin19220.htm"),
    "49170": ("Health Care Compliance", "Health and Human Services", "health/HealthCareCompliance49170.htm"),
    "49090": ("Health Care Technology", "Health and Human Services", "health/HlthCareTechnology49090.htm"),
    "19070": ("Hearing and Legal Services", "Administrative Services", "admin/HearingLegal19070.htm"),
    "29030": ("Historical Services and Preservation", "Educational and Media Services", "EducMediaServ/edu29030HistoricalServ.htm"),
    "79070": ("Housekeeping and Apparel Services", "Trades and Operations", "trades/HousekeepingApparelSvcs79070.htm"),
    "19090": ("Human Resources Services", "Administrative Services", "admin/HumanResource19090.htm"),
    "59070": ("Laboratory and Research Technicians and Specialists", "Natural Resources and Applied Science", "natresouc/nat59070LabResearch.htm"),
    "19110": ("Land Acquisition and Property Management", "Administrative Services", "admin/LandAcq19110.htm"),
    "69070": ("Law Enforcement", "Public Safety", "pubsafe/LawEnforcement69070.htm"),
    "29050": ("Library Services", "Educational and Media Services", "EducMediaServ/edu29050library.htm"),
    "59130": ("Life and Physical Science", "Natural Resources and Applied Science", "natresouc/nat59130LifePhysicalSci.htm"),
    "29070": ("Media and Production Services", "Educational and Media Services", "EducMediaServ/edu29070MediaProdSvcs.htm"),
    "59090": ("Minerals Regulatory Services", "Natural Resources and Applied Science", "natresouc/nat59090MineralsRegscvcs.htm"),
    "59110": ("Natural Resources Specialists", "Natural Resources and Applied Science", "natresouc/nat59110NatRes.htm"),
    "49110": ("Nursing/Physician Assistance Services", "Health and Human Services", "health/NursingPhysAsstance49110.htm"),
    "49130": ("Pharmaceutical Services", "Health and Human Services", "health/Pharma49130.htm"),
    "49150": ("Physician Services", "Health and Human Services", "health/Physiciansvcs49150.htm"),
    "19130": ("Policy Analysis and Planning", "Administrative Services", "admin/PolicyAnalysisPlan19130.htm"),
    "79090": ("Printing Operations", "Trades and Operations", "trades/PrintingOperations79090.htm"),
    "69090": ("Probation and Parole", "Public Safety", "pubsafe/ProbationParoleSvcs69090.htm"),
    "19210": ("Program Administration", "Administrative Services", "admin/ProgAdmin19210.htm"),
    "49210": ("Psychological Services", "Health and Human Services", "health/Psychologica49210.htm"),
    "29090": ("Public Relations and Marketing", "Educational and Media Services", "EducMediaServ/edu29090PublicRelationsMarketing.htm"),
    "69030": ("Public Safety Compliance", "Public Safety", "pubsafe/PublicSafetyCompliance69030.htm"),
    "49230": ("Rehabilitation Therapies", "Health and Human Services", "health/RehabTherapies49230.htm"),
    "79110": ("Retail Operations", "Trades and Operations", "trades/RetailOperations79110.htm"),
    "69110": ("Security Services", "Public Safety", "pubsafe/SecurityServices69110.htm"),
    "79130": ("Stores and Warehousing Operations", "Trades and Operations", "trades/StoresWareOperations79130.htm"),
    "29110": ("Training and Instruction", "Educational and Media Services", "EducMediaServ/edu29110Training.htm"),
    "79150": ("Transportation Operations", "Trades and Operations", "trades/TransportOperations79150.htm"),
    "79170": ("Utility Plant Operations", "Trades and Operations", "trades/UtilityPlantOper79170.htm"),
    "59150": ("Veterinary Science", "Natural Resources and Applied Science", "natresouc/nat59150VeterinaryScience.htm"),
}

# Build HTML_SOURCES list using f-string with the shared base URL
HTML_SOURCES = [
    (code, name, family, f"{HTML_BASE_URL}{path}", "html")
    for code, (name, family, path) in HTML_PATHS.items()
]

###### These are edge cases where the HTML source is not available or not suitable for scraping. Claude wanted to do this complicated hullabaloo to try to extract the URLs from the HTML pages, but it was just easier to hardcode these few PDF sources directly.


# PDF sources -- different hosting domains, so full URLs are stored directly
PDF_SOURCES = [
    ("59010", "Agricultural Services", "Natural Resources and Applied Science",
     "https://www.dhrm.virginia.gov/docs/default-source/compensationdocuments/agricultural-services-cgd-final44306F8E3605.pdf", "pdf"),
    ("79010", "Aircraft Operations", "Trades and Operations",
     "https://www.dhrm.virginia.gov/docs/default-source/hr/career-group-descriptions/aircraft-operations-cgd-final-docx.pdf", "pdf"),
    ("69150", "Emergency Services", "Public Safety",
     "https://web1.dhrm.virginia.gov/itech/DHRMWebAssets/careergroups/pubsafe/EmergencyServicesCareerGroup69150.pdf", "pdf"),
    ("39110", "Information Technology Specialists", "Engineering and Technology",
     "https://www.dhrm.virginia.gov/docs/default-source/compensationdocuments/info-tech-specialists-career-group-update.pdf", "pdf"),
    ("19150", "Procurement Services", "Administrative Services",
     "https://www.dhrm.virginia.gov/docs/default-source/compensationdocuments/procurement-services-career-group-description-update.pdf", "pdf"),
    ("79190", "Watercraft Operations", "Trades and Operations",
     "https://www.dhrm.virginia.gov/docs/default-source/hr/career-group-descriptions/79190-watercraft-operations-career-group.pdf", "pdf"),
]

ALL_SOURCES = HTML_SOURCES + PDF_SOURCES

## Claude original here, it makes the directories in my local PC.
# Create cache directories
CACHE_DIR = os.path.join("..", "data", "cache", "raw_pages")
os.makedirs(os.path.join(CACHE_DIR, "html"), exist_ok=True)
os.makedirs(os.path.join(CACHE_DIR, "pdf"), exist_ok=True)

# Phase 1 test subset - Financial Services only
TEST_SOURCES = [
    s for s in ALL_SOURCES if s[0] == "19030"
]

print(f"HTML base URL: {HTML_BASE_URL}")
print(f"HTML sources: {len(HTML_SOURCES)} (from {len(HTML_PATHS)} path entries)")
print(f"PDF sources:  {len(PDF_SOURCES)}")
print(f"Total:        {len(ALL_SOURCES)}")
print(f"Test subset:  {[s[1] for s in TEST_SOURCES]}")

HTML base URL: https://web1.dhrm.virginia.gov/itech/DHRMWebAssets/careergroups/
HTML sources: 50 (from 50 path entries)
PDF sources:  6
Total:        56
Test subset:  ['Financial Services']


## Scrape and Cache Raw Pages

Before we can parse any career group page, we need to download its raw content. To keep things
efficient and respectful, we use a **cache-first** strategy:

1. **Check the cache** - If we already saved the file to `raw_pages/html/` or `raw_pages/pdf/`,
   we skip the download and return the local path immediately.
2. **Fetch if missing** - If the file is not cached, we send an HTTP GET request and save the
   response to disk (text mode for HTML, binary mode for PDF).
3. **Rate-limit** - After every live fetch we pause for 1.5 seconds so we do not overwhelm
   the DHRM web server with rapid requests.

**Ethical scraping note:** All DHRM career group pages are public government data. We add a
polite delay between requests and cache locally so each page is downloaded at most once.

In [3]:
# ---------------------------------------------------------------------------
# Cell 3 - Fetch and Cache Function
# ---------------------------------------------------------------------------

def fetch_and_cache(url, source_format, career_group_code):
    """
    Download a DHRM career group page and save it to the local cache.

    Parameters
    ----------
    url : str
        Full URL of the career group page (HTML or PDF).
    source_format : str
        Either 'html' or 'pdf', controls the cache subdirectory and write mode.
    career_group_code : str
        The 5-digit career group code used as the filename.

    Returns
    -------
    str
        Local file path to the cached file.
    """
    ext = "htm" if source_format == "html" else "pdf"
    cache_path = os.path.join("..", "data", "cache", "raw_pages", source_format, f"{career_group_code}.{ext}")

    # Return cached version if it exists
    if os.path.exists(cache_path):
        print(f"  [CACHED] {career_group_code} - {cache_path}")
        return cache_path

    # Fetch from the web
    print(f"  [FETCH]  {career_group_code} - {url}")
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    # Save to cache
    if source_format == "html":
        with open(cache_path, "w", encoding="utf-8") as f:
            f.write(response.text)
    else:
        with open(cache_path, "wb") as f:
            f.write(response.content)

    # Polite delay before next request
    time.sleep(1.5)

    return cache_path

print("fetch_and_cache() defined.")

fetch_and_cache() defined.


## HTML Parsing Functions

Each DHRM career group page is parsed by five section-specific functions. The pages are old
Microsoft Word-generated HTML with no CSS classes or IDs, so all extraction is positional
and tag-based.

| Function | Section | Extracts |
|---|---|---|
| `parse_header(soup)` | Page header + effective date | Career group name, code, family, pay band range, concept of work |
| `parse_role_matrix(soup)` | Role matrix table | List of roles with code, name, pay band, and track |
| `parse_role_descriptions(soup, role_list)` | Role description blocks | Summary text and compensable factors (complexity, results, accountability) |
| `parse_soc_codes(soup, code)` | Statistical Reporting + role headers | SOC code mappings at group and role level |
| `parse_historical_titles(soup, role_list)` | History section | Former class codes, titles, and grades mapped to current roles |

In [4]:
# ---------------------------------------------------------------------------
# Cell 7 - parse_header(soup)
# ---------------------------------------------------------------------------

def parse_header(soup):
    """
    Extract career group header metadata from a DHRM career group page.
    
    Returns a dict with: career_group_name, career_group_code, occupational_family,
    pay_band_min, pay_band_max, occ_family_code, concept_of_work, effective_date.
    """
    result = {
        "career_group_name": None,
        "career_group_code": None,
        "occupational_family": None,
        "pay_band_min": None,
        "pay_band_max": None,
        "occ_family_code": None,
        "concept_of_work": None,
        "effective_date": None,
    }
    
    # --- Name and code from <TITLE> tag: "Financial Services #19030" ---
    title_text = soup.title.string if soup.title else ""
    title_match = re.match(r"(.+?)\s*#(\d+)", title_text)
    if title_match:
        result["career_group_name"] = title_match.group(1).strip()
        result["career_group_code"] = title_match.group(2).strip()
    
    # --- Occupational Family and Pay Band Range ---
    # Search full page text for pay band range (handles variant layouts).
    # Handles en-dash (Ãƒâ€šÃ¢â‚¬â€œ), plural "Pay Bands:", etc.
    full_text = re.sub(r"\s+", " ", soup.get_text())
    band_match = re.search(r"Pay Bands?(?:\s+Range)?:\s*(\d+)\s*[-–\s]\s*(\d+)", full_text)
    if band_match:
        result["pay_band_min"] = int(band_match.group(1))
        result["pay_band_max"] = int(band_match.group(2))
    for p_tag in soup.find_all("p"):
        p_text = p_tag.get_text()
        if "Occupational Family:" in p_text:
            fam_match = re.search(r"Occupational Family:\s*(.+?)(?:Pay Band|$)", re.sub(r"\s+", " ", p_text))
            if fam_match:
                result["occupational_family"] = fam_match.group(1).strip()
            break
    
    # --- occ_family_code = first 2 digits of career_group_code ---
    if result["career_group_code"]:
        result["occ_family_code"] = result["career_group_code"][:2]
    
    # --- Concept of Work: find the heading, then get the next content paragraph ---
    # Normalize whitespace because some pages split "Concept" and "of Work" across lines.
    for p_tag in soup.find_all("p"):
        if "Concept of Work" in re.sub(r"\s+", " ", p_tag.get_text()):
            # Walk forward through paragraphs looking for actual content.
            # Try JUSTIFY-aligned first, then any paragraph. Skip empty ones.
            sibling = p_tag
            for _ in range(10):
                sibling = sibling.find_next("p")
                if not sibling:
                    break
                text = sibling.get_text(separator=" ").strip()
                if text and len(text) > 20:
                    result["concept_of_work"] = re.sub(r"\s+", " ", text).strip()
                    break
            break
    
    # --- Effective Date from <li> tag ---
    for li_tag in soup.find_all("li"):
        li_text = li_tag.get_text(strip=True)
        date_match = re.search(r"(?:New\s+)?Effective Date:\s*(.+)", li_text)
        if date_match:
            result["effective_date"] = date_match.group(1).strip()
            break
    
    return result

print("parse_header() defined.")

parse_header() defined.


### Parser 2 -- Role Matrix (`parse_role_matrix`)

The **Role Matrix** is a table on each career group page that lists every role name, its
5-digit role code, and the pay band. Roles are organized into two tracks: **Practitioner**
(technical/specialist progression) and **Management** (supervisory progression). The parser
identifies the matrix table by its structure (rows containing pay band numbers in the first
column), then extracts each role with its code, band, and track designation.

In [5]:
# ---------------------------------------------------------------------------
# Cell 8 - parse_role_matrix(soup)
# ---------------------------------------------------------------------------

def parse_role_matrix(soup):
    """
    Extract the role matrix table listing all roles in this career group.
    
    Returns a list of dicts, each with: role_name, role_code, pay_band, track.
    Track is 'practitioner' or 'management' based on column position.
    """
    roles = []
    
    # Find the table containing "PAY BAND" and "PRACTITIONER" in header.
    # Normalize whitespace because some pages split these words across
    # separate <p> tags. Prefer leaf tables (no nested tables) to avoid
    # matching wrapper tables that contain the entire page.
    matrix_table = None
    for table in soup.find_all("table"):
        table_text = re.sub(r"\s+", " ", table.get_text())
        if "PAY BAND" in table_text and "PRACTITIONER" in table_text:
            if not table.find("table"):
                matrix_table = table
                break
    # Fallback: accept wrapper table if no leaf table matched
    if not matrix_table:
        for table in soup.find_all("table"):
            table_text = re.sub(r"\s+", " ", table.get_text())
            if "PAY BAND" in table_text and "PRACTITIONER" in table_text:
                matrix_table = table
                break
    
    if not matrix_table:
        return roles
    
    rows = matrix_table.find_all("tr")
    
    for row in rows[1:]:  # Skip header row
        cells = row.find_all("td")
        if len(cells) < 5:
            continue
        
        pay_band_text = cells[0].get_text(strip=True)
        if not pay_band_text.isdigit():
            continue
        pay_band = int(pay_band_text)
        
        # Practitioner role (columns 1-2)
        prac_name = re.sub(r"\s+", " ", cells[1].get_text(strip=True))
        prac_code = cells[2].get_text(strip=True)
        if prac_name and prac_name != "\xa0" and prac_code.isdigit():
            roles.append({
                "role_name": prac_name,
                "role_code": prac_code,
                "pay_band": pay_band,
                "track": "practitioner",
            })
        
        # Management role (columns 3-4)
        mgmt_name = re.sub(r"\s+", " ", cells[3].get_text(strip=True))
        mgmt_code = cells[4].get_text(strip=True)
        if mgmt_name and mgmt_name != "\xa0" and mgmt_code.isdigit():
            roles.append({
                "role_name": mgmt_name,
                "role_code": mgmt_code,
                "pay_band": pay_band,
                "track": "management",
            })
    
    return roles

print("parse_role_matrix() defined.")

parse_role_matrix() defined.


### Parsers 2 & 3 -- Role Descriptions (`parse_role_descriptions`)

After extracting the role matrix, we need each role's detailed description text. Every role
on the DHRM page has a dedicated section with a summary paragraph and three **compensable
factors** (Complexity, Results, Accountability). These factors describe the knowledge, skill,
and responsibility expectations at each level and are the primary text an LLM will match
against when classifying positions.

#### Three HTML Layout Variants

DHRM career group pages use three distinct HTML structures for role descriptions. The parser
detects which layout each page uses and applies the appropriate extraction strategy:

1. **Standard layout** (majority of pages) -- the header table has 4-cell rows
   (role name, Code, Pay Band, SOC), and the summary paragraph and factor table follow
   as sibling elements after the header table. Each factor table has three rows
   (COMPLEXITY, RESULTS, ACCOUNTABILITY).

2. **Embedded layout** (13 career groups, mostly 29xxx/59xxx ranges) -- the summary and
   factors are embedded as additional rows *inside* the header table rather than as
   siblings after it. The parser detects 2-cell factor rows within the table and
   extracts them in place.

3. **Transitional format** (career group 39010 only) -- the page has no per-role
   Code:/SOC: header tables at all. Role names appear as bold text in `<font>`
   elements. A dedicated handler matches role names from the role matrix to these
   elements and extracts the subsequent factor tables.

**Additional edge cases handled:** split factor tables (factors spread across 2-3 separate
tables instead of one), dual-track shared content (two roles sharing a single summary/factors
block), whitespace-only paragraphs containing non-breaking spaces, and factor label typos
in the source HTML (e.g., "COMPLELXITY", "ACCOUNTABIITY").

In [6]:
# ---------------------------------------------------------------------------
# Cell 9 - parse_role_descriptions(soup, role_list)
# ---------------------------------------------------------------------------
# Handles three HTML layout variants found across DHRM career group pages:
#   1. Standard layout   - summary + factors as siblings after header table
#   2. Embedded layout   - summary + factors as rows inside the header table
#   3. Transitional (39010) - no per-role header tables; role names in <font>
# Also handles split factor tables and dual-track shared content.
# ---------------------------------------------------------------------------

def _has_border(table):
    """Check if a table has the BORDER attribute (value may be empty string)."""
    return "border" in table.attrs and table.get("border") != "0"


def _extract_factor_content(tcells):
    """Extract factor text from a 2+ cell table row (label in cell 0, content in cell 1)."""
    items = [li.get_text(strip=True) for li in tcells[1].find_all("li")]
    content = " ".join(items) if items else re.sub(r"\s+", " ", tcells[1].get_text(strip=True))
    return content


def _match_factor_label(label):
    """Match a factor label, handling known typos in source HTML.
    Returns the canonical key or None."""
    label = label.upper().strip()
    # Exact and standard matches
    if "COMPLEXITY" in label or "COMPLELXITY" in label or "COMPLEXTIY" in label:
        return "complexity"
    if label in ("RESULTS", "RESULT"):
        return "results"
    if label.startswith("ACCOUNT"):
        # Handles ACCOUNTABILITY, ACCOUNTABIITY, ACCOUNTBILITY, etc.
        return "accountability"
    return None


def _extract_factors_from_paragraph(text):
    """Extract factor content from a paragraph that contains factor labels.
    Some pages embed COMPLEXITY/RESULTS/ACCOUNTABILITY as inline text in
    a <p> element rather than in table rows. The text typically has the pattern:
    COMPLEXITY\n content...\n RESULTS\n content...\n ACCOUNTABILITY\n content...
    """
    factors = {"complexity": None, "results": None, "accountability": None}
    # Use case-insensitive regex to split on factor labels while preserving
    # the original case of the content
    pattern = r'(?i)(COMPLE\w+|RESULTS?|ACCOUNT\w+)'
    parts = re.split(pattern, text)

    i = 0
    while i < len(parts):
        part_upper = parts[i].strip().upper()
        key = _match_factor_label(part_upper)
        if key and i + 1 < len(parts):
            content = re.sub(r"\s+", " ", parts[i + 1]).strip()
            if content:
                factors[key] = content
            i += 2
        else:
            i += 1
    return factors


def _extract_factors_from_table(table):
    """Extract all factor rows from a single table. Returns dict of found factors."""
    factors = {}
    for trow in table.find_all("tr"):
        tcells = trow.find_all("td")
        if len(tcells) >= 2:
            label = tcells[0].get_text(strip=True).upper()
            content = _extract_factor_content(tcells)
            key = _match_factor_label(label)
            if key:
                factors[key] = content
    return factors


def _is_header_table(table):
    """Check if a bordered table is a role header table (contains Code: and SOC:)."""
    if not _has_border(table):
        return False
    text = table.get_text()
    return "Code:" in text and "SOC:" in text


def _is_factor_table(table):
    """Check if a bordered table contains compensable factor rows."""
    if not _has_border(table):
        return False
    text = table.get_text().upper()
    return ("COMPLE" in text or "RESULTS" in text or "RESULT" in text
            or "ACCOUNT" in text)


def _is_real_text(text):
    """Check if text is non-empty after stripping whitespace and non-breaking spaces."""
    if text is None:
        return False
    cleaned = text.replace("\xa0", " ").strip()
    return len(cleaned) > 0


def _extract_embedded_content(rows):
    """
    Extract summary and factors from non-4-cell rows inside an embedded header table.

    In embedded layout, the header table contains:
      - 4-cell rows: role headers (role name, code, pay band, SOC)
      - 1-cell rows: role name sub-headers or summary text
      - 2-cell rows: factor rows (COMPLEXITY, RESULTS, ACCOUNTABILITY)

    For dual-track headers, the pattern repeats for each role:
      [role name sub-header] [summary] [factors...] [role name sub-header] [summary] [factors...]

    Returns a list of (summary, factors_dict) tuples, one per content block.
    """
    blocks = []
    current_summary = None
    current_factors = {"complexity": None, "results": None, "accountability": None}
    in_block = False

    for row in rows:
        cells = row.find_all("td")
        ncells = len(cells)

        if ncells == 4:
            # This is a header row, skip it
            continue

        if ncells == 1:
            # Could be a role name sub-header or a summary paragraph
            text = re.sub(r"\s+", " ", cells[0].get_text(separator=" ")).strip()
            text = text.replace("\xa0", " ").strip()

            if not text:
                continue

            # If we already have factors accumulated, save the block
            if in_block and any(v is not None for v in current_factors.values()):
                blocks.append((current_summary, dict(current_factors)))
                current_summary = None
                current_factors = {"complexity": None, "results": None, "accountability": None}
                in_block = False

            # Check if this looks like a summary (starts with "The" and is long)
            # or a role name sub-header (short, matches a role name pattern)
            if text.lower().startswith("the") and len(text) > 40:
                current_summary = text
                in_block = True
            else:
                # Role name sub-header -- start a new block
                in_block = True

        elif ncells == 2:
            label = cells[0].get_text(strip=True).upper()
            content = _extract_factor_content(cells)
            key = _match_factor_label(label)
            if key:
                current_factors[key] = content
                in_block = True

    # Save the last block
    if in_block and any(v is not None for v in current_factors.values()):
        blocks.append((current_summary, dict(current_factors)))

    return blocks


def _walk_siblings_for_content(start_sibling, header_tables_set):
    """
    Walk sibling elements after a header table to find summary + factors.

    Handles:
    - Standard layout: one summary <p> followed by one or more factor tables
    - Split factor tables: COMPLEXITY in one table, RESULTS/ACCOUNTABILITY in next
    - Skips empty/whitespace-only paragraphs

    Returns (summary_text, factors_dict, next_sibling) where next_sibling is where
    we stopped (for continuing with the next role in dual-track headers).
    """
    current = start_sibling
    summary_text = None
    factors = {"complexity": None, "results": None, "accountability": None}

    # Phase 1: Find the summary paragraph
    # Uses the same text extraction as the original parser to preserve exact
    # formatting (including leading spaces) for regression compatibility.
    while current is not None:
        if hasattr(current, "name") and current.name:
            if current.name == "p":
                align = (current.get("align") or "").upper()
                # Match original parser: get_text(separator=" ") + collapse whitespace
                raw_txt = re.sub(r"\s+", " ", current.get_text(separator=" "))
                clean_txt = raw_txt.replace("\xa0", " ").strip()

                if align in ("JUSTIFY", "") and _is_real_text(clean_txt):
                    if len(clean_txt) > 30:
                        # Use the raw text (with possible leading space) to
                        # preserve the original parser's exact output format
                        summary_text = raw_txt
                        current = current.next_sibling
                        break

                # Skip empty or short paragraphs
                if not _is_real_text(clean_txt):
                    current = current.next_sibling
                    continue

                # Check for centered role name sub-headers (skip them)
                if align == "CENTER" and len(clean_txt) < 80:
                    current = current.next_sibling
                    continue

                # If it has real text that might be a summary
                if len(clean_txt) > 30:
                    summary_text = raw_txt
                    current = current.next_sibling
                    break

            if current.name == "table" and _has_border(current):
                tbl_text = current.get_text().upper()
                if any(k in tbl_text for k in ["COMPLE", "RESULTS", "RESULT", "ACCOUNT"]):
                    break  # Factor table found before summary
                if "CODE:" in current.get_text() and "SOC:" in current.get_text():
                    break  # Next header table
        current = current.next_sibling

    # Phase 2: Collect factors from one or more consecutive factor tables.
    # Some pages have extra paragraphs between the summary and the factor
    # table, or between split factor tables. We only stop at a paragraph
    # if we have already found at least one factor (meaning we're past the
    # current role's content and hitting the next role's summary).
    found_any_factor = False
    while current is not None:
        if hasattr(current, "name") and current.name:
            if current.name == "table" and _has_border(current):
                # Stop at the next header table
                if current in header_tables_set:
                    break

                tbl_text = current.get_text().upper()
                if any(k in tbl_text for k in ["COMPLE", "RESULTS", "RESULT", "ACCOUNT"]):
                    table_factors = _extract_factors_from_table(current)
                    for key, val in table_factors.items():
                        if val is not None and factors[key] is None:
                            factors[key] = val
                            found_any_factor = True
                    current = current.next_sibling
                    continue
                else:
                    # Non-factor table (e.g., historical titles) -- stop
                    break

            elif current.name == "p":
                txt_upper = current.get_text().upper()
                # Check if this paragraph contains factor content
                # (some pages embed COMPLEXITY/RESULTS/ACCOUNTABILITY
                # directly in a <p> element instead of a table)
                if (not found_any_factor
                        and any(k in txt_upper for k in
                                ["COMPLE", "RESULT", "ACCOUNT"])):
                    p_factors = _extract_factors_from_paragraph(
                        current.get_text())
                    for key, val in p_factors.items():
                        if val is not None and factors[key] is None:
                            factors[key] = val
                            found_any_factor = True
                    current = current.next_sibling
                    continue

                txt = current.get_text(strip=True).replace("\xa0", "").strip()
                # Only stop at a paragraph if we already found factors
                # (meaning this paragraph is the next role's summary)
                if found_any_factor and txt and len(txt) > 30:
                    break
                # Skip empty/short paragraphs and pre-factor paragraphs

        current = current.next_sibling

    return summary_text, factors, current


def _handle_transitional_39010(soup, role_list):
    """
    Special handler for career group 39010 (Computer Operations).

    This page has no per-role Code:/SOC: header tables. Instead, role names
    appear as bold text within <font> elements, followed by summary text and
    factor tables, all inside a single outer table cell.

    The role names in the page use "Technician" while role_list uses "Tech",
    so we use fuzzy matching. Names are sorted longest-first to avoid
    "Technician I" matching inside "Technician II".
    """
    # Build index: normalize role names for fuzzy matching
    name_to_idx = {}
    for i, r in enumerate(role_list):
        name = re.sub(r"\s+", " ", r["role_name"]).strip()
        name_to_idx[name] = i
        # Also add expanded form (Tech -> Technician)
        expanded = name.replace("Tech I", "Technician I").replace("Tech II", "Technician II")
        if expanded != name:
            name_to_idx[expanded] = i

    # Sort names longest-first to avoid partial matches
    sorted_names = sorted(name_to_idx.keys(), key=len, reverse=True)

    # Find the main content cell that contains role descriptions
    content_cell = None
    for td in soup.find_all("td"):
        text = td.get_text()
        if "COMPLEXITY" in text and "Role Descriptions" in text:
            content_cell = td
            break

    if not content_cell:
        return role_list

    # Walk children of the content cell
    # Pattern: <font> with role name + summary, followed by factor table(s)
    children = list(content_cell.children)
    in_role_section = False
    current_role_idx = None
    current_summary = None

    i = 0
    while i < len(children):
        ch = children[i]

        if not hasattr(ch, "name") or not ch.name:
            i += 1
            continue

        if ch.name in ("b", "font"):
            text = re.sub(r"\s+", " ", ch.get_text(strip=True)).strip()

            if not text:
                i += 1
                continue

            # Check for "Statistical Reporting" to stop
            if "Statistical Reporting" in text:
                break

            # Check if this starts a role description section
            # Use longest-match-first to avoid partial matches
            role_match = None
            for rname in sorted_names:
                if rname in text:
                    role_match = (rname, name_to_idx[rname])
                    break

            if role_match:
                rname, ridx = role_match
                in_role_section = True
                current_role_idx = ridx

                # Extract summary: text after the role name
                name_end = text.index(rname) + len(rname)
                after_name = text[name_end:].strip()
                if after_name and len(after_name) > 20:
                    current_summary = after_name
                elif text.lower().startswith("this role") or \
                     ("role" in text.lower() and len(text) > 40):
                    current_summary = text
                else:
                    current_summary = None

                i += 1
                continue

            # If we're in a role section and text looks like a summary
            if in_role_section and current_role_idx is not None:
                if text.lower().startswith("this role") or \
                   (len(text) > 40 and "role" in text.lower()):
                    current_summary = text
                    if not role_list[current_role_idx].get("role_summary"):
                        role_list[current_role_idx]["role_summary"] = text

            i += 1
            continue

        elif ch.name == "table" and in_role_section:
            border = ch.get("border", "NONE")
            if border != "0" and "border" in ch.attrs:
                tbl_text = ch.get_text().upper()
                if "CLASS" in tbl_text:
                    # Historical titles table -- stop role section
                    break

                if any(k in tbl_text for k in
                       ["COMPLE", "RESULTS", "RESULT", "ACCOUNT"]):
                    table_factors = _extract_factors_from_table(ch)

                    if current_role_idx is not None:
                        if current_summary and \
                                not role_list[current_role_idx].get("role_summary"):
                            role_list[current_role_idx]["role_summary"] = \
                                current_summary

                        for key, val in table_factors.items():
                            if val is not None:
                                existing = role_list[current_role_idx].get(key)
                                if existing is None:
                                    role_list[current_role_idx][key] = val

                    # Continue to pick up split factor tables
                    i += 1
                    continue

        i += 1

    return role_list


def parse_role_descriptions(soup, role_list):
    """
    Enrich each role dict with role_summary, complexity, results, accountability.

    Handles three layout variants:
    1. Standard layout: header table with 4-cell rows, followed by sibling
       summary paragraph and factor table(s).
    2. Embedded layout: header table with 4-cell header rows AND additional
       1-cell (summary) and 2-cell (factor) rows inside the same table.
    3. Transitional (39010): no per-role header tables; role content in <font>
       elements inside a single outer table cell.

    Also handles:
    - Split factor tables (factors spread across 2-3 consecutive tables)
    - Dual-track headers (2+ roles sharing content blocks)
    - Empty/whitespace-only paragraphs (including non-breaking spaces)
    - "RESULT" (singular) label variant
    """
    # Build lookups from role_code and role_name to role dict index
    code_to_idx = {r["role_code"]: i for i, r in enumerate(role_list)}
    name_to_idx = {re.sub(r"\s+", " ", r["role_name"]).strip(): i
                   for i, r in enumerate(role_list)}
    # Normalized name lookup for fuzzy matching (handles whitespace differences
    # between role_list names and header table names)
    norm_name_to_idx = {_normalize_for_match(r["role_name"]): i
                        for i, r in enumerate(role_list)}

    # No-space name lookup for cases with inconsistent word breaks
    nospace_name_to_idx = {_normalize_nospace(r["role_name"]): i
                          for i, r in enumerate(role_list)}

    def _resolve_idx(role_name, role_code):
        """Resolve a role to its index using name, code, or normalized name.
        Falls back to no-space matching for cases where source HTML has
        different word breaks than the role matrix (e.g., 'andRepair' vs
        'and Repair')."""
        idx = name_to_idx.get(role_name)
        if idx is None:
            idx = code_to_idx.get(role_code)
        if idx is None:
            norm = _normalize_for_match(role_name)
            idx = norm_name_to_idx.get(norm)
        if idx is None:
            # Try with all whitespace removed (handles inconsistent word breaks)
            nospace = _normalize_nospace(role_name)
            idx = nospace_name_to_idx.get(nospace)
        if idx is None:
            # Last resort: try substring matching on normalized names
            norm = _normalize_for_match(role_name)
            for stored_norm, stored_idx in norm_name_to_idx.items():
                if norm in stored_norm or stored_norm in norm:
                    idx = stored_idx
                    break
        return idx

    # Find all "header tables" - bordered tables whose text contains
    # "Code:" and "SOC:" (standard and embedded layouts)
    header_tables = []
    for table in soup.find_all("table"):
        if not _has_border(table):
            continue
        text = table.get_text()
        if "Code:" in text and "SOC:" in text:
            header_tables.append(table)

    # If no header tables found, try the transitional (39010) handler
    if not header_tables:
        return _handle_transitional_39010(soup, role_list)

    # Build a set for fast lookup when walking siblings
    header_tables_set = set(header_tables)

    for ht in header_tables:
        rows = ht.find_all("tr")

        # Extract role entries from 4-cell rows
        role_entries = []  # list of (code, name) tuples
        non_header_rows = []  # rows that are NOT 4-cell headers
        for row in rows:
            cells = row.find_all("td")
            if len(cells) == 4:
                role_name_raw = re.sub(r"\s+", " ",
                                       cells[0].get_text(strip=True)).strip()
                code_text = cells[1].get_text(strip=True)
                code_match = re.search(r"Code:\s*(\d+)", code_text)
                if code_match:
                    role_entries.append((code_match.group(1), role_name_raw))
                else:
                    non_header_rows.append(row)
            else:
                non_header_rows.append(row)

        if not role_entries:
            continue

        # ------------------------------------------------------------------
        # Detect layout: if there are non-header rows with factor content
        # inside this table, use embedded extraction
        # ------------------------------------------------------------------
        has_embedded_factors = False
        for row in non_header_rows:
            cells = row.find_all("td")
            if len(cells) == 2:
                label = cells[0].get_text(strip=True).upper()
                if _match_factor_label(label) is not None:
                    has_embedded_factors = True
                    break

        if has_embedded_factors:
            # ----- EMBEDDED LAYOUT -----
            blocks = _extract_embedded_content(non_header_rows)

            if len(role_entries) == 1:
                # Single role: all blocks belong to it
                code, name = role_entries[0]
                idx = _resolve_idx(name, code)
                if idx is not None:
                    # Merge all blocks (there may be multiple sub-role
                    # descriptions but they all map to the same role code)
                    for summary, factors in blocks:
                        if summary and not role_list[idx].get("role_summary"):
                            role_list[idx]["role_summary"] = summary
                        for key in ("complexity", "results", "accountability"):
                            if factors.get(key) and not role_list[idx].get(key):
                                role_list[idx][key] = factors[key]

            else:
                # Multiple roles in one embedded header table.
                # The non-header rows contain sub-sections for each role,
                # identified by 1-cell role name sub-headers.
                # We need to map blocks to role entries.
                #
                # Strategy: walk non_header_rows, tracking which role we
                # are currently in based on 1-cell role name matches.
                _assign_embedded_multi_role(
                    non_header_rows, role_entries, role_list,
                    name_to_idx, code_to_idx, norm_name_to_idx
                )

        else:
            # ----- STANDARD LAYOUT (sibling walk) -----
            current = ht.next_sibling
            entry_idx = 0

            while entry_idx < len(role_entries) and current is not None:
                role_code, role_name = role_entries[entry_idx]

                idx = _resolve_idx(role_name, role_code)

                summary, factors, current = _walk_siblings_for_content(
                    current, header_tables_set
                )

                if idx is not None:
                    if summary:
                        role_list[idx]["role_summary"] = summary
                    for key in ("complexity", "results", "accountability"):
                        if factors.get(key):
                            role_list[idx][key] = factors[key]

                entry_idx += 1

            # ----- DUAL-TRACK SHARED CONTENT -----
            # If some roles in this header got nothing, share data from
            # the role(s) that did get content.
            if len(role_entries) > 1:
                # Find a role entry that has all factors populated
                donor_idx = None
                for code, name in role_entries:
                    ridx = _resolve_idx(name, code)
                    if ridx is not None:
                        r = role_list[ridx]
                        if (r.get("role_summary") and r.get("complexity")
                                and r.get("results") and r.get("accountability")):
                            donor_idx = ridx
                            break

                if donor_idx is not None:
                    donor = role_list[donor_idx]
                    for code, name in role_entries:
                        ridx = _resolve_idx(name, code)
                        if ridx is not None and ridx != donor_idx:
                            r = role_list[ridx]
                            if not r.get("role_summary"):
                                r["role_summary"] = donor["role_summary"]
                            for key in ("complexity", "results", "accountability"):
                                if not r.get(key):
                                    r[key] = donor[key]

    return role_list


def _normalize_for_match(text):
    """Collapse all whitespace to single spaces and lowercase for matching."""
    return re.sub(r"\s+", " ", text).strip().lower()


def _normalize_nospace(text):
    """Remove ALL whitespace and lowercase for aggressive matching."""
    return re.sub(r"\s+", "", text).strip().lower()


def _assign_embedded_multi_role(non_header_rows, role_entries, role_list,
                                 name_to_idx, code_to_idx,
                                 norm_name_to_idx=None):
    """
    For embedded tables with multiple roles, walk non-header rows and assign
    summary + factors to each role based on 1-cell role name sub-headers.

    The pattern in these tables is:
      [1-cell: Role Name A]
      [1-cell: Summary for Role A]
      [2-cell: COMPLEXITY ...]
      [2-cell: RESULTS ...]
      [2-cell: ACCOUNTABILITY ...]
      [1-cell: Role Name B (or sub-variant like "Specialist III (Interpreter)")]
      [1-cell: Summary for sub-variant]
      [2-cell: COMPLEXITY ...]
      ...
      [1-cell: Role Name B (Manager I)]
      [1-cell: Summary for Manager I]
      [2-cell: COMPLEXITY ...]
      ...
    """
    # Build normalized role name -> (code, name) lookup
    # Use both the role_entry names (from header cells) and role_list names
    role_name_set = {}
    for code, name in role_entries:
        norm = _normalize_for_match(name)
        role_name_set[norm] = (code, name)

    # Also add role_list names (may differ in whitespace from header cell names)
    for i, r in enumerate(role_list):
        norm = _normalize_for_match(r["role_name"])
        if norm not in role_name_set:
            # Only add if the code matches one of our entries
            for code, name in role_entries:
                if r["role_code"] == code:
                    role_name_set[norm] = (code, name)
                    break

    # Walk rows, tracking current role
    current_idx = None

    for row in non_header_rows:
        cells = row.find_all("td")
        ncells = len(cells)

        if ncells == 1:
            text = re.sub(r"\s+", " ", cells[0].get_text(separator=" ")).strip()
            text = text.replace("\xa0", " ").strip()

            if not text:
                continue

            # Distinguish summaries from role name sub-headers.
            # Summaries typically start with "The" and are long (>60 chars).
            # Role name sub-headers are shorter and don't start with "The".
            text_lower = text.lower()
            is_summary_like = (
                text_lower.startswith("the") and len(text) > 60
            )

            if is_summary_like:
                # This is a summary paragraph
                if current_idx is None:
                    # No role matched yet -- assign to the first role entry
                    first_code, first_name = role_entries[0]
                    current_idx = (name_to_idx.get(first_name)
                                   or code_to_idx.get(first_code))
                    if current_idx is None and norm_name_to_idx:
                        current_idx = norm_name_to_idx.get(
                            _normalize_for_match(first_name))
                if current_idx is not None:
                    if not role_list[current_idx].get("role_summary"):
                        role_list[current_idx]["role_summary"] = text
                continue

            # Check if this 1-cell row matches a role name (role sub-header)
            text_norm = _normalize_for_match(text)
            matched_role = False
            for norm_name, (code, name) in role_name_set.items():
                # Exact match or one contains the other (after normalization)
                if norm_name == text_norm or norm_name in text_norm or text_norm in norm_name:
                    new_idx = name_to_idx.get(name) or code_to_idx.get(code)
                    if new_idx is None and norm_name_to_idx:
                        new_idx = norm_name_to_idx.get(
                            _normalize_for_match(name))
                    if new_idx is not None:
                        current_idx = new_idx
                        matched_role = True
                        break

            if not matched_role and current_idx is not None:
                # Not a role name match -- check if this is a summary
                if len(text) > 30:
                    if not role_list[current_idx].get("role_summary"):
                        role_list[current_idx]["role_summary"] = text

        elif ncells == 2 and current_idx is not None:
            label = cells[0].get_text(strip=True).upper()
            content = _extract_factor_content(cells)
            key = _match_factor_label(label)

            if key and content and not role_list[current_idx].get(key):
                role_list[current_idx][key] = content

print("parse_role_descriptions() defined.")


parse_role_descriptions() defined.


### Parser 4 -- SOC Codes (`parse_soc_codes`)

Each career group page includes a **Statistical Reporting** section listing Bureau of Labor
Statistics Standard Occupational Classification (SOC) codes. These federal codes link DHRM
roles to national occupational standards used in Workday, Banner, and other HR systems. The
parser locates the SOC section by searching for its heading text, then extracts each code-title
pair using a regex pattern that matches the `XX-XXXX` format.

In [7]:
# ---------------------------------------------------------------------------
# Cell 10 - parse_soc_codes(soup, career_group_code)
# ---------------------------------------------------------------------------

def parse_soc_codes(soup, career_group_code):
    """
    Extract SOC code mappings at both group and role level.
    
    Group-level: from the Statistical Reporting table (SOC code + title).
    Role-level: from each role's header table (4th cell contains "SOC: XX-XXXX*").
    
    Returns a list of dicts with: career_group_code, role_code, soc_code,
    soc_title, mapping_level, data_source.
    """
    soc_entries = []
    
    # --- Group-level SOC codes from Statistical Reporting section ---
    stat_heading = None
    for p_tag in soup.find_all("p"):
        if "Statistical Reporting" in p_tag.get_text():
            stat_heading = p_tag
            break
    
    if stat_heading:
        soc_table = stat_heading.find_next("table")
        if soc_table:
            for row in soc_table.find_all("tr"):
                cells = row.find_all("td")
                if len(cells) >= 2:
                    soc_code = cells[0].get_text(strip=True)
                    soc_title = re.sub(r"\s+", " ", cells[1].get_text(strip=True))
                    # Clean up SOC code: strip trailing decimals
                    soc_code = re.sub(r"\.\d+$", "", soc_code)
                    if re.match(r"\d{2}-\d{4}$", soc_code):
                        soc_entries.append({
                            "career_group_code": career_group_code,
                            "role_code": None,
                            "soc_code": soc_code,
                            "soc_title": soc_title,
                            "mapping_level": "career_group",
                            "data_source": "DHRM_HTML",
                        })
    
    # --- Role-level SOC codes from role header tables ---
    for table in soup.find_all("table"):
        if not _has_border(table):
            continue
        text = table.get_text()
        if "Code:" not in text or "SOC:" not in text:
            continue
        
        for row in table.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) != 4:
                continue
            code_text = cells[1].get_text(strip=True)
            soc_text = cells[3].get_text(strip=True)
            
            code_match = re.search(r"Code:\s*(\d+)", code_text)
            soc_match = re.search(r"SOC:\s*([\d-]+)", soc_text)
            
            if code_match and soc_match:
                raw_soc = soc_match.group(1)
                # Clean: strip trailing decimals, take first valid XX-XXXX code
                raw_soc = re.sub(r"\.\d+$", "", raw_soc)
                # Only add if it matches standard XX-XXXX format
                if re.match(r"\d{2}-\d{4}$", raw_soc):
                    soc_entries.append({
                        "career_group_code": career_group_code,
                        "role_code": code_match.group(1),
                        "soc_code": raw_soc,
                        "soc_title": None,
                        "mapping_level": "role",
                        "data_source": "DHRM_HTML",
                    })
    
    return soc_entries

print("parse_soc_codes() defined.")

parse_soc_codes() defined.


### Parser 5 -- Historical Titles (`parse_historical_titles`)

DHRM career group pages end with a **History of Classes** section that maps former state
classification codes and titles to their current role equivalents. These mappings are critical
for identifying legacy positions that need reclassification. The parser walks the history table
row by row, tracking which role each set of historical titles belongs to, and extracts the
former class code, title text, and pay grade.

In [8]:
# ---------------------------------------------------------------------------
# Cell 11 - parse_historical_titles(soup, role_list)
# ---------------------------------------------------------------------------

def parse_historical_titles(soup, role_list):
    """
    Extract historical class titles from the History section.
    
    Each role name appears as a bold heading followed by a bordered table
    with CLASS CODE | CLASS TITLE | GRADE columns.
    
    Returns a list of dicts with: role_code, former_class_code,
    former_class_title, former_grade, data_source.
    """
    hist_entries = []
    
    # Build name-to-code lookup (normalize whitespace for matching)
    name_to_code = {}
    for r in role_list:
        normalized = re.sub(r"\s+", " ", r["role_name"]).strip()
        name_to_code[normalized] = r["role_code"]
    
    # Find the History section heading in the page.
    # Prefer <p> tags (the actual section heading) over <a> tags (nav links).
    # Computer Operations uses <p align="CENTER"><a name="history"></a>History</p>
    history_heading = None
    for tag in soup.find_all("p"):
        if tag.get_text(strip=True) == "History":
            history_heading = tag
            break
    if not history_heading:
        for tag in soup.find_all(["b", "h2", "h3", "h4"]):
            if tag.get_text(strip=True) == "History":
                history_heading = tag
                break
    
    if not history_heading:
        return hist_entries
    
    # Find all CLASS CODE tables after the History heading.
    # For each table, look backwards through preceding elements
    # to find the bold role name heading that precedes it.
    # Match tables with CLASS/CODE/TITLE/GRADE headers (text may have
    # newlines between CLASS and CODE on some pages like Computer Operations)
    history_tables = [
        t for t in history_heading.find_all_next("table")
        if "CLASS" in t.get_text() and "TITLE" in t.get_text() and "GRADE" in t.get_text()
    ]
    
    for table in history_tables:
        # Find the role name by searching backwards through bold and
        # paragraph elements for a known role name
        role_code = None
        for prev_tag in table.find_all_previous(["b", "p", "a"]):
            prev_text = re.sub(r"\s+", " ", prev_tag.get_text(strip=True)).strip()
            if prev_text in name_to_code:
                role_code = name_to_code[prev_text]
                break
        
        if not role_code:
            continue
        
        rows = table.find_all("tr")
        for row in rows[1:]:  # Skip header row
            cells = row.find_all("td")
            if len(cells) >= 3:
                class_code = cells[0].get_text(strip=True)
                class_title = re.sub(r"\s+", " ", cells[1].get_text(strip=True))
                grade = cells[2].get_text(strip=True)
                
                if class_code and class_title:
                    hist_entries.append({
                        "role_code": role_code,
                        "former_class_code": class_code,
                        "former_class_title": class_title,
                        "former_grade": grade,
                        "data_source": "DHRM_HTML",
                    })
    
    return hist_entries

print("parse_historical_titles() defined.")

parse_historical_titles() defined.


### Parser Test -- Financial Services Reference Check

Before scaling to all 50 HTML pages, we test the five section parsers against a single known
career group -- Financial Services (#19030). This page was used during development as the
reference data source, so we know the expected outputs. If the parsers handle this page correctly,
they should generalize to the rest of the HTML pages that share the same template.

In [9]:
# ---------------------------------------------------------------------------
# Cell 12 - Quick test of all parsers against Financial Services (#19030)
# ---------------------------------------------------------------------------

with open(os.path.join("..", "data", "cache", "raw_pages", "html", "19030.htm"), "r", encoding="utf-8") as f:
    test_soup = BeautifulSoup(f.read(), "html.parser")

# Test parse_header
header = parse_header(test_soup)
print("=== Header ===")
for k, v in header.items():
    val_preview = str(v)[:80] if v else "None"
    print(f"  {k}: {val_preview}")

# Test parse_role_matrix
roles = parse_role_matrix(test_soup)
print(f"\n=== Role Matrix ({len(roles)} roles, expected 7) ===")
for r in roles:
    print(f"  {r['role_code']} | {r['role_name']} | Band {r['pay_band']} | {r['track']}")

# Test parse_role_descriptions
roles = parse_role_descriptions(test_soup, roles)
print(f"\n=== Role Descriptions ===")
for r in roles:
    has_summary = 'Yes' if r.get('role_summary') else 'MISSING'
    has_complex = 'Yes' if r.get('complexity') else 'MISSING'
    has_results = 'Yes' if r.get('results') else 'MISSING'
    has_account = 'Yes' if r.get('accountability') else 'MISSING'
    print(f"  {r['role_code']} | Summary:{has_summary} | C:{has_complex} | R:{has_results} | A:{has_account}")

# Test parse_soc_codes
socs = parse_soc_codes(test_soup, "19030")
print(f"\n=== SOC Codes ({len(socs)} entries, expected 12: 5 group + 7 role) ===")
for s in socs:
    level = s['mapping_level']
    role = s['role_code'] or 'GROUP'
    print(f"  [{level}] {role} -> {s['soc_code']} {s.get('soc_title', '') or ''}")

# Test parse_historical_titles
hist = parse_historical_titles(test_soup, roles)
print(f"\n=== Historical Titles ({len(hist)} entries, expected 59) ===")
from collections import Counter
role_counts = Counter(t['role_code'] for t in hist)
for code, count in sorted(role_counts.items()):
    role_name = next((r['role_name'] for r in roles if r['role_code'] == code), code)
    print(f"  {code} ({role_name}): {count} titles")

=== Header ===
  career_group_name: Financial Services
  career_group_code: 19030
  occupational_family: Administrative Services
  pay_band_min: 4
  pay_band_max: 8
  occ_family_code: 19
  concept_of_work: This Career Group provides career tracks for financial specialists who perform, 
  effective_date: 11/01/01

=== Role Matrix (7 roles, expected 7) ===
  19031 | Financial Services Specialist I | Band 4 | practitioner
  19032 | Financial Services Specialist II | Band 5 | practitioner
  19034 | Financial Services Manager I | Band 5 | management
  19033 | Financial Services Specialist III | Band 6 | practitioner
  19035 | Financial Services Manager II | Band 6 | management
  19036 | Financial Services Manager III | Band 7 | management
  19037 | Financial Services Manager IV | Band 8 | management

=== Role Descriptions ===
  19031 | Summary:Yes | C:Yes | R:Yes | A:Yes
  19032 | Summary:Yes | C:Yes | R:Yes | A:Yes
  19034 | Summary:Yes | C:Yes | R:Yes | A:Yes
  19033 | Summary:Yes | C:Yes

## PDF Career Group Parser (`parse_pdf_page`)

While 50 of the 56 DHRM career group pages are published as HTML, six career groups
are only available as PDF documents. HTML and PDF require fundamentally different
parsing strategies:

- **HTML parsing** (BeautifulSoup) navigates a structured DOM tree where tables,
  headings, and paragraphs are explicitly tagged. The parser locates data by CSS
  selectors and tag hierarchy.
- **PDF parsing** works with a flat stream of positioned text fragments. There is no
  DOM -- the parser must reconstruct structure from the spatial layout of characters
  on each page.

We use **pdfplumber** for PDF text extraction. pdfplumber reads the PDF's internal
content streams and reassembles characters into lines based on their x/y coordinates,
producing clean plain text that preserves the visual reading order. This is more
reliable than raw `PyPDF2` text extraction for the two-column compensable-factor
tables in these career group documents.

The `parse_pdf_page()` function mirrors the same interface contract as
`parse_html_page()` -- it accepts a filepath, career group code, name, and
occupational family, and returns the same four-tuple of `(header, role_list,
soc_list, hist_list)`. This allows the orchestrator loop to call either parser
transparently based on the source format.

**PDF-specific parsing challenges handled:**
- Dual-career-track roles where two `Code:` headers appear consecutively and share
  a single content block with role-name sub-headings
- Role names split across lines (e.g., "Information Technology" / "Specialist I")
- SOC codes spanning multiple continuation lines (e.g., "SOC: 53-5021\*, 53-5022\*,"
  / "and 53-5011\*")
- Page headers/footers injected into the text stream
- "SOC: see appendix" references requiring group-level SOC lookup


In [10]:
# ---------------------------------------------------------------------------
# Cell 13 - PDF Career Group Parser
# ---------------------------------------------------------------------------

def parse_pdf_page(filepath, career_group_code, career_group_name, occupational_family):
    """
    Parse a DHRM career group PDF and return structured data matching the
    HTML parser's interface contract.

    Parameters
    ----------
    filepath : str
        Path to the cached PDF file.
    career_group_code : str
        Five-digit career group code (e.g. '59010').
    career_group_name : str
        Career group display name (e.g. 'Agricultural Services').
    occupational_family : str
        Occupational family name (e.g. 'Natural Resources and Applied Sciences').

    Returns
    -------
    header : dict
        Career group metadata.
    role_list : list[dict]
        One dict per role with compensable factors.
    soc_list : list[dict]
        SOC code mappings (group-level from the Statistical Reporting section).
    hist_list : list[dict]
        Historical class title mappings.
    """
    # ------------------------------------------------------------------
    # 1. Extract full text from all PDF pages
    # ------------------------------------------------------------------
    full_text = _extract_full_text(filepath)

    # ------------------------------------------------------------------
    # 2. Parse header block (name, family, pay band range, concept of work)
    # ------------------------------------------------------------------
    header = _parse_header(full_text, career_group_code, career_group_name,
                           occupational_family)

    # ------------------------------------------------------------------
    # 3. Parse role descriptions with compensable factors
    # ------------------------------------------------------------------
    role_list = _parse_roles(full_text, career_group_code)

    # ------------------------------------------------------------------
    # 4. Parse SOC codes from the Statistical Reporting section
    # ------------------------------------------------------------------
    soc_list = _parse_soc_section(full_text, career_group_code, role_list)

    # ------------------------------------------------------------------
    # 5. Parse historical class titles
    # ------------------------------------------------------------------
    hist_list = _parse_history(full_text, role_list)

    return header, role_list, soc_list, hist_list


# ======================================================================
# Internal helpers
# ======================================================================

def _extract_full_text(filepath):
    """Read every page of the PDF and concatenate text."""
    pages = []
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n".join(pages)


def _clean_text(text):
    """Strip page headers/footers and date stamps."""
    # Remove page header lines (e.g. "Emergency Services #69150 Page 3 of 10")
    text = re.sub(r'^.*#\d{5}\s+Page\s+\d+\s+of\s+\d+\s*$', '',
                  text, flags=re.MULTILINE)
    # Remove standalone date stamps (e.g. "8/17/2015 1")
    text = re.sub(r'^\d{1,2}/\d{1,2}/\d{4}\s+\d+\s*$', '',
                  text, flags=re.MULTILINE)
    # Remove standalone page numbers
    text = re.sub(r'^\d{1,2}\s*$', '', text, flags=re.MULTILINE)
    return text


def _parse_header(full_text, career_group_code, career_group_name,
                  occupational_family):
    """Extract pay band range and concept of work from the document top."""
    header = {
        'career_group_code': career_group_code,
        'career_group_name': career_group_name,
        'occupational_family': occupational_family,
        'occ_family_code': career_group_code[:2],
        'pay_band_min': None,
        'pay_band_max': None,
        'concept_of_work': None,
        'data_source': 'DHRM_PUBLIC',
    }

    # Pay band range -- "Pay Band Range: X - Y" or "Pay Bands: X - Y"
    pb_match = re.search(
        r'Pay\s+Band(?:s|[\s]+Range):\s*(\d+)\s*[-\u2013]\s*(\d+)', full_text)
    if pb_match:
        header['pay_band_min'] = int(pb_match.group(1))
        header['pay_band_max'] = int(pb_match.group(2))

    # Concept of Work -- text between heading and "Roles Comprising"
    cow_match = re.search(
        r'Concept\s+of\s+Work\s*\n(.*?)(?=Roles\s+Comprising)',
        full_text, re.DOTALL
    )
    if cow_match:
        cow = cow_match.group(1).strip()
        cow = re.sub(r'\s+', ' ', cow)
        header['concept_of_work'] = cow

    return header


# ------------------------------------------------------------------
# Role parsing
# ------------------------------------------------------------------

def _parse_roles(full_text, career_group_code):
    """
    Parse individual role descriptions including SOC codes from headers,
    pay band, and compensable factors.

    Strategy: first collect all Code: header lines to build the role
    skeleton, then find each role's content block by searching for the
    role name sub-heading followed by COMPLEXITY/RESULTS/ACCOUNTABILITY.
    """
    text = _clean_text(full_text)
    lines = text.split('\n')

    # ----- Step 1: Collect role header lines -----
    role_header_pat = re.compile(
        r'^(.+?)\s+Code:\s*(\d{5})\s+Pay\s+Band:\s*(\d+)\s+SOC:\s*(.+?)$',
        re.MULTILINE
    )

    raw_roles = []
    for m in role_header_pat.finditer(text):
        role_name_raw = m.group(1).strip()
        role_code = m.group(2)
        pay_band = int(m.group(3))
        soc_fragment = m.group(4).strip()

        # Find the line number for this header
        header_line_num = _find_line_num(lines, role_code, pay_band)

        # Gather SOC continuation lines
        soc_text = soc_fragment
        if header_line_num is not None:
            for cl in range(header_line_num + 1,
                            min(header_line_num + 4, len(lines))):
                cont = lines[cl].strip()
                if re.match(r'^(?:and\s+)?\d{2}-\d{4}', cont):
                    soc_text += ' ' + cont
                elif cont.startswith('and '):
                    soc_text += ' ' + cont
                else:
                    break

        # Handle "see appendix" (39110)
        if 'see' in soc_text.lower() or 'appendix' in soc_text.lower():
            soc_text = ''

        # Check for name continuation on the next line
        if header_line_num is not None:
            role_name_raw = _complete_role_name(
                role_name_raw, lines, header_line_num, soc_text)

        # Parse SOC codes from header
        role_socs = re.findall(r'(\d{2}-\d{4})', soc_text)

        # Determine track
        track = 'management' if 'Manager' in role_name_raw else 'practitioner'

        raw_roles.append({
            'role_code': role_code,
            'role_name': role_name_raw,
            'career_group_code': career_group_code,
            'pay_band': pay_band,
            'track': track,
            'role_summary': None,
            'complexity': None,
            'results': None,
            'accountability': None,
            'data_source': 'DHRM_PUBLIC',
            '_soc_codes': role_socs,
            '_header_pos': m.start(),
        })

    # ----- Step 2: Find content blocks for each role -----
    # The content for each role starts with the role name as a sub-heading
    # followed by "The <RoleName> role provides..." and compensable factors.
    # For dual-track roles, the content is NOT between consecutive Code:
    # headers but rather under the role name sub-heading in a shared block.

    # Find the end of the role descriptions section
    stat_idx = text.find('Statistical Reporting')
    hist_idx = _find_history_section_start(text)
    section_end = len(text)
    for b in [stat_idx, hist_idx]:
        if b > 0:
            section_end = min(section_end, b)

    # For each role, find its content by looking for the role name sub-heading
    # followed by COMPLEXITY marker
    for role in raw_roles:
        block = _find_role_content_block(text, role, raw_roles, section_end)
        if block:
            summary, complexity, results, accountability = \
                _parse_compensable_block(block, role['role_name'])
            role['role_summary'] = summary
            role['complexity'] = complexity
            role['results'] = results
            role['accountability'] = accountability

    return raw_roles


def _find_line_num(lines, role_code, pay_band):
    """Find the line number containing a specific Code: and Pay Band:."""
    for i, line in enumerate(lines):
        if f'Code: {role_code}' in line and f'Pay Band: {pay_band}' in line:
            return i
    return None


def _complete_role_name(name, lines, header_line_num, soc_text):
    """
    Check if the next line after a role header contains a name continuation
    (e.g. 'Specialist I', 'Manager II') and append it.
    """
    if header_line_num + 1 >= len(lines):
        return name

    next_line = lines[header_line_num + 1].strip()
    if not next_line:
        return name

    # Skip if it's a SOC continuation
    if re.match(r'^(?:and\s+)?\d{2}-\d{4}', next_line):
        return name

    # Skip if it starts a description paragraph
    if next_line.startswith('The '):
        return name

    # Skip "appendix" (part of "SOC: see appendix")
    if next_line.lower() == 'appendix':
        return name

    # Name continuations: "Specialist I", "Manager II", etc.
    # May have trailing "appendix" from "SOC: see / appendix" split
    name_cont_match = re.match(
        r'^((?:Specialist|Manager|Coordinator|Officer|Operator)\s+[IVX]+)'
        r'(?:\s+appendix)?\s*$',
        next_line)
    if name_cont_match:
        return name + ' ' + name_cont_match.group(1)

    # "Manager II" that appears as a standalone continuation (79190)
    manager_match = re.match(r'^(Manager\s+[IVX]+)\s*$', next_line)
    if manager_match:
        return name + ' ' + manager_match.group(1)

    return name


def _find_role_content_block(text, role, all_roles, section_end):
    """
    Find the actual content block for a role by searching for the role name
    as a sub-heading, followed by the description paragraph and compensable
    factors.

    For non-dual-track roles, this is simply the text between the role's
    Code: header and the next Code: header. For dual-track roles, the
    content is within a shared block, separated by role name sub-headings.
    """
    role_name = role['role_name']
    role_code = role['role_code']
    header_pos = role['_header_pos']

    # Find the next role header position (or section end)
    next_positions = [r['_header_pos'] for r in all_roles
                      if r['_header_pos'] > header_pos]
    block_end = min(next_positions) if next_positions else section_end

    # Get the text between this header and the next
    between = text[header_pos:block_end]

    # Check if COMPLEXITY appears in this block
    if 'COMPLEXITY' in between:
        # Content is in this block. For dual-track, we may need to split.
        # Look for the role name sub-heading within the block
        return _extract_role_from_block(between, role_name, role_code)

    # Content might be in a later block (dual-track: two consecutive headers
    # followed by shared content). Search forward from the last header in the
    # consecutive group.
    # Find the block that contains our role's content
    search_start = block_end
    # The next block should contain our role name sub-heading
    next_next_positions = [r['_header_pos'] for r in all_roles
                           if r['_header_pos'] > search_start]
    search_end = min(next_next_positions) if next_next_positions else section_end

    extended = text[search_start:search_end]
    if role_name in extended:
        return _extract_role_from_block(extended, role_name, role_code)

    # Broadest search: look in the entire role descriptions section
    role_desc_start = text.find('Role Descriptions')
    if role_desc_start < 0:
        role_desc_start = 0
    full_section = text[role_desc_start:section_end]
    return _extract_role_from_block(full_section, role_name, role_code)


def _extract_role_from_block(block, role_name, role_code):
    """
    Within a text block, find the content for a specific role by looking
    for the role name sub-heading and its COMPLEXITY section.
    """
    # Escape the role name for regex
    escaped = re.escape(role_name)

    # Look for the role name appearing as a sub-heading (standalone or before
    # "The ... role provides"). In dual-track blocks, the pattern is:
    # "RoleName\nThe RoleName role provides..."
    # or "RoleName\nCOMPLEXITY ..."

    # Find all positions where this role name appears
    role_name_positions = [m.start() for m in re.finditer(escaped, block)]

    best_start = None
    low_start = None
    code_fallback = False
    for pos in role_name_positions:
        # Check what follows this occurrence
        after = block[pos + len(role_name):pos + len(role_name) + 200]
        after_stripped = after.lstrip()
        # Two-pass: prefer high-confidence matches (Code: header,
        # 'The ' summary start, 'role ' mid-summary) over low-confidence
        # matches (sub-heading before COMPLEXITY) which lose the summary.
        is_high = (after_stripped.startswith('Code:') or
                   after_stripped.startswith('The ') or
                   after_stripped.startswith('role '))
        is_low = (after_stripped.startswith('\n') or
                  after_stripped.startswith('COMPLEXITY') or
                  after_stripped.startswith('\r'))
        if is_high or is_low:
            before = block[max(0, pos - 10):pos]
            if 'Code:' not in before:
                if is_high:
                    best_start = pos
                    break  # High confidence -- use immediately
                elif low_start is None:
                    low_start = pos  # Save but keep searching


    if best_start is None:
        # Try Code: header fallback (preserves summary text)
        code_pat = re.search(rf'Code:\s*{role_code}', block)
        if code_pat:
            best_start = code_pat.end()
            code_fallback = True
        elif low_start is not None:
            best_start = low_start  # Sub-heading match (may lose summary)
        else:
            return block  # Return entire block as fallback


    # Find the end of this role's content
    # Ends at the next role name sub-heading or end of block
    remaining = block[best_start:] if code_fallback else block[best_start + len(role_name):]

    # Look for the next role sub-heading (a known role name on its own line)
    # Simple heuristic: look for ACCOUNTABILITY section, then the next
    # standalone capitalized line that isn't a compensable factor label
    acc_idx = remaining.find('ACCOUNTABILITY')
    if acc_idx > 0:
        after_acc = remaining[acc_idx:]
        # Find where ACCOUNTABILITY's content ends
        # Look for the next role-like heading or section boundary
        end_markers = [
            r'\nThese\s+(?:two|three)\s+roles',
            r'\n(?:Agricultural|Information|Emergency|Pilot|Procurement|Watercraft)\s+(?:Specialist|Manager|Coordinator|Officer|Operations|Technology)',
            r'\nStatistical\s+Reporting',
            r'\nHistory\b',
            r'\nNew\s+Effective\s+Date',
            r'\nRevised\s',
            r'\nFinal\s',
        ]
        earliest_end = len(remaining)
        for marker in end_markers:
            em = re.search(marker, after_acc)
            if em:
                candidate = acc_idx + em.start()
                if candidate < earliest_end:
                    earliest_end = candidate
        remaining = remaining[:earliest_end]

    return remaining


def _parse_compensable_block(block, role_name):
    """
    Extract role summary and three compensable factors from a role's
    text block. Returns (summary, complexity, results, accountability).
    """
    # Find COMPLEXITY marker
    complexity_idx = block.find('COMPLEXITY')

    # Role summary = text before COMPLEXITY
    if complexity_idx > 0:
        summary_text = block[:complexity_idx].strip()
    else:
        summary_text = block.strip()

    # Extract the summary paragraph starting with "The"
    the_match = re.search(r'(The\s+.+?)(?=\n\s*COMPLEXITY|\n\s*$|\Z)',
                          summary_text, re.DOTALL)
    if the_match:
        summary = re.sub(r'\s+', ' ', the_match.group(1)).strip()
    else:
        summary = re.sub(r'\s+', ' ', summary_text).strip()
        if not summary or len(summary) < 20:
            summary = None

    # Extract compensable factors
    complexity = _extract_factor(block, 'COMPLEXITY', 'RESULTS')
    results = _extract_factor(block, 'RESULTS', 'ACCOUNTABILITY')
    accountability = _extract_factor_last(block, 'ACCOUNTABILITY')

    return summary, complexity, results, accountability


def _extract_factor(text, start_label, end_label):
    """Extract compensable factor text between two labels."""
    start_idx = text.find(start_label)
    end_idx = text.find(end_label, start_idx + 1) if start_idx >= 0 else -1
    if start_idx < 0:
        return None
    if end_idx < 0:
        end_idx = len(text)
    section = text[start_idx + len(start_label):end_idx]
    return _collect_bullets(section)


def _extract_factor_last(text, label):
    """Extract the last compensable factor (ACCOUNTABILITY)."""
    idx = text.find(label)
    if idx < 0:
        return None
    section = text[idx + len(label):]
    return _collect_bullets(section)


def _collect_bullets(section):
    """
    From a compensable factor section, collect bullet-point content
    while skipping the left-column definition text.
    """
    bullets = []
    # Definition paragraphs to skip (left column in the PDF table)
    skip_patterns = re.compile(
        r'^(?:Describes\s|resources\s|processes\s|difficulty\s|'
        r'assignments|contacts|terms\s+of|guidance|into\s+account|'
        r'encountered|work,\s+scope|KSA|nature\s+of|benefit\s+or|'
        r'gain\s+or|goodwill|account\s+impact|effect\s+of|'
        r'consequence|authority\s+exercised|autonomy\s+of|'
        r'finality\s+of|leadership,\s+judgment|judgment\s+and|'
        r'independence\s+of|range\s+and\s+impact)',
        re.IGNORECASE
    )

    for line in section.split('\n'):
        stripped = line.strip()
        if not stripped:
            continue
        # Bullet points (the actual factor descriptions)
        if stripped.startswith('\u2022'):
            bullets.append(stripped.lstrip('\u2022 ').strip())
        elif skip_patterns.match(stripped):
            continue
        else:
            # Continuation of previous bullet or standalone text
            if bullets:
                bullets[-1] = bullets[-1] + ' ' + stripped
            else:
                bullets.append(stripped)

    if not bullets:
        return None
    result = ' '.join(bullets)
    return re.sub(r'\s+', ' ', result).strip() or None


# ------------------------------------------------------------------
# SOC section parsing
# ------------------------------------------------------------------

def _parse_soc_section(full_text, career_group_code, role_list):
    """
    Parse the Statistical Reporting section for group-level SOC codes,
    plus collect role-level SOC codes from role headers.
    """
    soc_list = []

    # Role-level SOC codes from headers
    for role in role_list:
        for soc_code in role.get('_soc_codes', []):
            soc_list.append({
                'career_group_code': career_group_code,
                'role_code': role['role_code'],
                'soc_code': soc_code,
                'soc_title': '',
                'mapping_level': 'role',
                'data_source': 'DHRM_PUBLIC',
            })

    # Group-level SOC codes from Statistical Reporting section
    stat_idx = full_text.find('Statistical Reporting')
    if stat_idx < 0:
        stat_idx = full_text.find('Standard Occupational Classification')
    if stat_idx < 0:
        return soc_list

    hist_idx = _find_history_section_start(full_text, stat_idx)
    end_idx = hist_idx if hist_idx > 0 else len(full_text)
    soc_section = full_text[stat_idx:end_idx]

    # Extract unique SOC codes with their titles
    seen_socs = set()
    for soc_code in re.findall(r'(\d{2}-\d{4})', soc_section):
        if soc_code in seen_socs:
            continue
        seen_socs.add(soc_code)
        title = _find_soc_title(soc_section, soc_code)
        soc_list.append({
            'career_group_code': career_group_code,
            'role_code': None,
            'soc_code': soc_code,
            'soc_title': title,
            'mapping_level': 'group',
            'data_source': 'DHRM_PUBLIC',
        })

        # Back-fill titles for role-level entries
        for entry in soc_list:
            if (entry['soc_code'] == soc_code and
                    entry['mapping_level'] == 'role' and
                    not entry['soc_title']):
                entry['soc_title'] = title

    return soc_list


def _find_soc_title(section, soc_code):
    """Find the occupational title for a SOC code in the SOC section."""
    idx = section.find(soc_code)
    if idx < 0:
        return ''

    before = section[:idx]
    before_lines = before.split('\n')

    title_parts = []
    for line in reversed(before_lines[-3:]):
        stripped = line.strip()
        if not stripped:
            continue
        if re.search(r'\d{2}-\d{4}', stripped):
            break
        if len(stripped) > 100:
            break
        title_parts.insert(0, stripped)
        if len(title_parts) >= 2:
            break

    title = ' '.join(title_parts).strip()
    title = re.sub(r'\s{2,}.*', '', title)
    if ',' in title:
        parts = title.split(',')
        title = ','.join(parts[:2]).strip().rstrip(',')
    return title


# ------------------------------------------------------------------
# History parsing
# ------------------------------------------------------------------

def _find_history_section_start(text, search_from=0):
    """Find the position of the History section header."""
    for m in re.finditer(r'\bHistory\b', text[search_from:]):
        pos = search_from + m.start()
        after = text[pos:pos + 500]
        if re.search(r'(?:Previous\s+class|Previous\s+Role|'
                     r'This\s+Career\s+Group\s+Description|'
                     r'CLASS\s+CODE|CLASS\s+TITLE)',
                     after, re.IGNORECASE):
            return pos
    return -1


def _parse_history(full_text, role_list):
    """
    Parse the History section to extract former class codes, titles,
    and grades.
    """
    hist_list = []

    hist_idx = _find_history_section_start(full_text)
    if hist_idx < 0:
        return hist_list

    hist_section = _clean_text(full_text[hist_idx:])

    # Build mapping of role names to role codes
    role_name_to_code = {}
    for role in role_list:
        role_name_to_code[role['role_name']] = role['role_code']

    lines = hist_section.split('\n')
    current_role_code = None

    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue

        # Skip section header
        if stripped == 'History':
            continue
        # Skip preamble lines (various forms)
        if stripped.startswith('Previous class') or \
                stripped.startswith('Previous Role'):
            continue
        if stripped.startswith('This Career Group Description'):
            continue
        # Skip multi-line preamble continuation (69150 has a long
        # merger note spanning several lines)
        if current_role_code is None and not re.match(r'^[A-Z]', stripped):
            continue

        # Skip CLASS/CODE table headers
        if re.match(r'^CLASS\b', stripped, re.IGNORECASE):
            continue
        if re.match(r'^CODE\b', stripped, re.IGNORECASE):
            continue

        # Skip date/revision lines
        if re.match(r'^(?:Revised|Final|New Effective)\s', stripped):
            continue
        # Skip footnotes
        if stripped.startswith('*'):
            continue

        # Check for role name header
        matched_role = _match_role_name(stripped, role_name_to_code)
        if matched_role:
            current_role_code = matched_role
            continue

        # Skip "Previous Role Titles" sub-section and its table
        if 'Previous Role Titles' in stripped:
            current_role_code = None
            continue
        if re.match(r'^Role\s+Role\s+Title\s+Pay', stripped):
            continue
        if re.match(r'^Code\s+Band', stripped):
            continue

        # Skip preamble continuation lines before first role match
        if current_role_code is None:
            continue

        # Parse class history entry: "XXXXX Title Words Grade"
        # Grade can be numeric (3, 10, 20/21), alpha (TR), or omitted
        class_match = re.match(
            r'^(\d{5})\s+(.+?)\s+(\d+(?:/\d+)?|[A-Z]{2})\s*$', stripped)
        if class_match:
            hist_list.append({
                'role_code': current_role_code,
                'former_class_code': class_match.group(1),
                'former_class_title': class_match.group(2).strip(),
                'former_grade': class_match.group(3),
                'data_source': 'DHRM_PUBLIC',
            })

    return hist_list


def _match_role_name(line, role_name_to_code):
    """
    Check if a line in the History section matches a known role name.
    Returns the role_code if matched, else None.
    """
    normalized = re.sub(r'\s+', ' ', line).strip()

    # Exact match
    for name, code in role_name_to_code.items():
        if re.sub(r'\s+', ' ', name).strip() == normalized:
            return code

    # Partial/substring match
    for name, code in role_name_to_code.items():
        norm_name = re.sub(r'\s+', ' ', name).strip()
        if norm_name in normalized or normalized in norm_name:
            return code

    return None


print(f"parse_pdf_page() defined -- ready for {len(PDF_SOURCES)} PDF career groups.")


parse_pdf_page() defined -- ready for 6 PDF career groups.


## Stage 4 - Orchestrator and Execution Loop

The `parse_html_page()` function ties together all five section parsers into a single call that processes one career-group HTML page end to end. It opens the cached file, builds a BeautifulSoup tree, and calls each parser in dependency order:

1. **parse_header** - extracts the career-group-level metadata (name, pay band range, concept of work, effective date).
2. **parse_role_matrix** - returns the list of roles with codes, pay bands, and career tracks.
3. **parse_role_descriptions** - enriches each role with its narrative fields (summary, complexity, results, accountability).
4. **parse_soc_codes** - collects the SOC crosswalk entries at both the group and role level.
5. **parse_historical_titles** - maps former classification titles to their current role codes.

The execution loop iterates over a configurable source list. For Phase 1 testing we use `TEST_SOURCES` (Financial Services only). Switching to `HTML_SOURCES` or `ALL_SOURCES` scales the pipeline to the full DHRM catalog without changing any logic.

In [11]:
# ---------------------------------------------------------------------------
# Cell 14 - Orchestrator function + execution loop
# ---------------------------------------------------------------------------

def parse_html_page(filepath, career_group_code, career_group_name, occupational_family):
    """Orchestrate all section parsers for one HTML career group page."""
    with open(filepath, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    header = parse_header(soup)
    # Override with known values from source list (more reliable than parsing)
    header['career_group_code'] = career_group_code
    header['career_group_name'] = career_group_name
    header['occupational_family'] = occupational_family
    header['occ_family_code'] = career_group_code[:2]

    role_list = parse_role_matrix(soup)
    role_list = parse_role_descriptions(soup, role_list)
    soc_list  = parse_soc_codes(soup, career_group_code)
    hist_list = parse_historical_titles(soup, role_list)

    return header, role_list, soc_list, hist_list


# ---- Phase 2: All HTML sources ----
# Change to HTML_SOURCES for Phase 2, ALL_SOURCES for full run
sources_to_process = ALL_SOURCES

all_career_groups    = []
all_roles            = []
all_soc_codes        = []
all_historical_titles = []
errors               = []

for code, name, family, url, fmt in sources_to_process:
    print(f"\nProcessing: {name} ({code})")
    try:
        filepath = fetch_and_cache(url, fmt, code)

        if fmt == "html":
            header, roles, socs, hist = parse_html_page(filepath, code, name, family)
        else:
            header, roles, socs, hist = parse_pdf_page(filepath, code, name, family)

        # Add metadata
        header['source_url']    = url
        header['source_format'] = fmt
        header['data_source']   = 'DHRM_PUBLIC'

        # Add career_group_code to each role for FK linkage
        for r in roles:
            r['career_group_code'] = code
            r['data_source']       = 'DHRM_PUBLIC'

        all_career_groups.append(header)
        all_roles.extend(roles)
        all_soc_codes.extend(socs)
        # Remove internal fields from PDF parser roles
        for r in roles:
            r.pop('_soc_codes', None)
            r.pop('_header_pos', None)

        all_historical_titles.extend(hist)

        print(f"  OK: {len(roles)} roles, {len(socs)} SOC codes, {len(hist)} historical titles")

    except Exception as e:
        print(f"  ERROR: {e}")
        errors.append((code, name, str(e)))

print(f"\n{'='*50}")
print(f"Career groups:    {len(all_career_groups)}")
print(f"Roles:            {len(all_roles)}")
print(f"SOC codes:        {len(all_soc_codes)}")
print(f"Historical titles: {len(all_historical_titles)}")
if errors:
    print(f"ERRORS: {len(errors)}")
    for code, name, err in errors:
        print(f"  {code} {name}: {err}")


Processing: Administration and Office Support (19010)
  [CACHED] 19010 - raw_pages\html\19010.htm
  OK: 3 roles, 22 SOC codes, 40 historical titles

Processing: Architecture and Engineering Services (39050)
  [CACHED] 39050 - raw_pages\html\39050.htm
  OK: 6 roles, 28 SOC codes, 48 historical titles

Processing: Audit and Management Services (19190)
  [CACHED] 19190 - raw_pages\html\19190.htm


  OK: 7 roles, 9 SOC codes, 26 historical titles

Processing: Building Trades (79030)
  [CACHED] 79030 - raw_pages\html\79030.htm


  OK: 6 roles, 70 SOC codes, 58 historical titles

Processing: Computer Operations (39010)


  [CACHED] 39010 - raw_pages\html\39010.htm


  OK: 4 roles, 0 SOC codes, 11 historical titles

...

  OK: 3 roles, 5 SOC codes, 5 historical titles

Processing: Emergency Services (69150)
  [CACHED] 69150 - raw_pages\pdf\69150.pdf


  OK: 6 roles, 16 SOC codes, 23 historical titles

Processing: Information Technology Specialists (39110)
  [CACHED] 39110 - raw_pages\pdf\39110.pdf


  OK: 7 roles, 22 SOC codes, 87 historical titles

Processing: Procurement Services (19150)
  [CACHED] 19150 - raw_pages\pdf\19150.pdf


  OK: 7 roles, 9 SOC codes, 23 historical titles

Processing: Watercraft Operations (79190)
  [CACHED] 79190 - raw_pages\pdf\79190.pdf


  OK: 4 roles, 13 SOC codes, 12 historical titles

Career groups:    56
Roles:            294
SOC codes:        666
Historical titles: 1469


## Stage 5 - Data Validation

Before loading parsed data into a database we convert the raw lists of dictionaries into pandas DataFrames and run five categories of quality checks:

1. **Code pattern checks** - career_group_code and role_code must be exactly five digits (`^\d{5}$`); SOC codes must follow the BLS format `XX-XXXX` (`^\d{2}-\d{4}$`).
2. **Pay band range check** - every pay_band value must fall between 1 and 10 (the Commonwealth pay structure).
3. **Referential integrity** - every role record's `career_group_code` must match a row in the career_groups table (parent-child FK).
4. **Required fields** - critical columns (career_group_code, career_group_name, concept_of_work, role_code, role_name) must not contain nulls.
5. **Reference data comparison** - for the Financial Services test page we know the expected counts (7 roles, 12 SOC codes, 59 historical titles), so we flag any deviation.

Any failures are printed as warnings rather than exceptions so the pipeline continues and the analyst can review all issues at once.

In [12]:
# ---------------------------------------------------------------------------
# Cell 16 - Data validation
# ---------------------------------------------------------------------------
import re

df_career_groups    = pd.DataFrame(all_career_groups)
df_roles            = pd.DataFrame(all_roles)
df_soc_codes        = pd.DataFrame(all_soc_codes)
df_historical_titles = pd.DataFrame(all_historical_titles)

validation_pass = True

print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)

# 1. Code pattern checks
print("\n1. Code Pattern Checks")
bad_cg = df_career_groups[~df_career_groups['career_group_code'].astype(str).str.match(r'^\d{5}$')]
bad_rc = df_roles[~df_roles['role_code'].astype(str).str.match(r'^\d{5}$')]
bad_soc = df_soc_codes[~df_soc_codes['soc_code'].astype(str).str.match(r'^\d{2}-\d{4}$')]
if len(bad_cg) == 0 and len(bad_rc) == 0 and len(bad_soc) == 0:
    print("   PASS - All codes match expected patterns")
else:
    validation_pass = False
    if len(bad_cg) > 0:
        print(f"   FAIL - {len(bad_cg)} career group codes do not match ^\\d{{5}}$")
    if len(bad_rc) > 0:
        print(f"   FAIL - {len(bad_rc)} role codes do not match ^\\d{{5}}$")
    if len(bad_soc) > 0:
        print(f"   FAIL - {len(bad_soc)} SOC codes do not match ^\\d{{2}}-\\d{{4}}$")

# 2. Pay band range check
print("\n2. Pay Band Range Check (1-10)")
bands = pd.to_numeric(df_roles['pay_band'], errors='coerce')
out_of_range = bands[(bands < 1) | (bands > 10)]
if len(out_of_range) == 0:
    print("   PASS - All pay bands within 1-10")
else:
    validation_pass = False
    print(f"   FAIL - {len(out_of_range)} pay bands outside range 1-10")

# 3. Referential integrity
print("\n3. Referential Integrity (roles -> career_groups)")
cg_codes = set(df_career_groups['career_group_code'].astype(str))
orphan_roles = df_roles[~df_roles['career_group_code'].astype(str).isin(cg_codes)]
if len(orphan_roles) == 0:
    print("   PASS - Every role links to a valid career group")
else:
    validation_pass = False
    print(f"   FAIL - {len(orphan_roles)} roles reference missing career groups")

# 4. Required fields
print("\n4. Required Fields (no nulls)")
req_cg = ['career_group_code', 'career_group_name', 'concept_of_work']
req_role = ['role_code', 'role_name']
nulls_found = False
for col in req_cg:
    n = df_career_groups[col].isna().sum()
    if n > 0:
        print(f"   FAIL - career_groups.{col} has {n} null(s)")
        nulls_found = True
for col in req_role:
    n = df_roles[col].isna().sum()
    if n > 0:
        print(f"   FAIL - roles.{col} has {n} null(s)")
        nulls_found = True
if not nulls_found:
    print("   PASS - No nulls in required fields")
else:
    validation_pass = False

# 5. Financial Services reference check (known counts for 19030)
print("\n5. Financial Services Reference Check (19030)")
fs_roles = df_roles[df_roles['career_group_code'] == '19030']
fs_socs  = df_soc_codes[df_soc_codes['career_group_code'] == '19030']
fs_hist  = df_historical_titles[df_historical_titles['role_code'].isin(fs_roles['role_code'])]
expected = {'roles': 7, 'soc_codes': 12, 'hist_titles': 59}
actual   = {'roles': len(fs_roles), 'soc_codes': len(fs_socs), 'hist_titles': len(fs_hist)}
for label in expected:
    status = "PASS" if actual[label] == expected[label] else "WARN"
    if status == "WARN":
        validation_pass = False
    print(f"   {status} - {label}: expected {expected[label]}, got {actual[label]}")

print(f"\n{'='*60}")
print(f"OVERALL: {'ALL CHECKS PASSED' if validation_pass else 'ISSUES FOUND - review above'}")
print(f"{'='*60}")

# Sample data preview
print("\n--- Career Groups Sample ---")
print(df_career_groups[['career_group_code', 'career_group_name', 'occupational_family',
                         'pay_band_min', 'pay_band_max']].to_string(index=False))
print("\n--- Roles Sample (first 5) ---")
print(df_roles[['career_group_code', 'role_code', 'role_name', 'pay_band', 'track']].head().to_string(index=False))

DATA VALIDATION REPORT

1. Code Pattern Checks
   PASS - All codes match expected patterns

2. Pay Band Range Check (1-10)
   PASS - All pay bands within 1-10

3. Referential Integrity (roles -> career_groups)
   PASS - Every role links to a valid career group

4. Required Fields (no nulls)
   PASS - No nulls in required fields

5. Financial Services Reference Check (19030)
   PASS - roles: expected 7, got 7
   PASS - soc_codes: expected 12, got 12
   PASS - hist_titles: expected 59, got 59

OVERALL: ALL CHECKS PASSED

--- Career Groups Sample ---
career_group_code                                   career_group_name                   occupational_family  pay_band_min  pay_band_max
            19010                   Administration and Office Support               Administrative Services             1             3
            39050               Architecture and Engineering Services            Engineering and Technology             5             8
            19190                     

## Salary Structure Reference Table

DHRM publishes a **Classified Salary Structure** that defines the minimum and maximum salary
for each pay band. This data connects the abstract pay band numbers in our roles table to
actual dollar ranges.

We parse the FY26 salary structure PDF (effective June 10, 2025) using pdfplumber, extracting
only the **Virginia Statewide Pay Area (SW)** bands. The Northern Virginia Pay Area uses
expanded ranges that are not relevant to our statewide classification tool.

The resulting `dhrm_pay_bands` DataFrame becomes a fifth reference table alongside career groups,
roles, SOC codes, and historical titles.

In [13]:
# ---------------------------------------------------------------------------
# Salary Structure - Parse FY26 Statewide Pay Bands
# ---------------------------------------------------------------------------
import pdfplumber
import re

SALARY_PDF_URL = "https://www.dhrm.virginia.gov/docs/default-source/compensationdocuments/fy26salarystructure.pdf"
salary_pdf_path = os.path.join("..", "data", "cache", "raw_pages", "pdf", "fy26_salary_structure.pdf")

# Download if not cached
if not os.path.exists(salary_pdf_path):
    print("Downloading FY26 salary structure PDF...")
    r = requests.get(SALARY_PDF_URL, timeout=30)
    r.raise_for_status()
    with open(salary_pdf_path, "wb") as f:
        f.write(r.content)
    print(f"  Saved ({len(r.content)} bytes)")
else:
    print(f"[CACHED] {salary_pdf_path}")

# Extract statewide pay bands from the PDF
with pdfplumber.open(salary_pdf_path) as pdf:
    page_text = pdf.pages[0].extract_text()

# Parse the Statewide (SW) section only -- stop before Northern Virginia
sw_section = page_text.split("Northern Virginia")[0]

# Extract band rows: "1 $28,360 $65,631"
band_rows = re.findall(
    r"(\d+)\s+\$([0-9,]+)\s+(\$[0-9,]+|Market)",
    sw_section
)

salary_data = []
for band_str, min_str, max_str in band_rows:
    salary_data.append({
        "pay_band": int(band_str),
        "pay_area": "SW",
        "minimum_salary": int(min_str.replace(",", "")),
        "maximum_salary": int(max_str.replace("$", "").replace(",", "")) if max_str != "Market" else None,
        "effective_date": "2025-06-10",
        "fiscal_year": "FY26",
        "data_source": "DHRM_PUBLIC",
    })

df_dhrm_pay_bands = pd.DataFrame(salary_data)

print(f"\nFY26 Statewide Salary Structure ({len(df_dhrm_pay_bands)} bands):")
print(df_dhrm_pay_bands.to_string(index=False))

[CACHED] raw_pages/pdf/fy26_salary_structure.pdf

FY26 Statewide Salary Structure (9 bands):
 pay_band pay_area  minimum_salary  maximum_salary effective_date fiscal_year data_source
        1       SW           28360         65631.0     2025-06-10        FY26 DHRM_PUBLIC
        2       SW           30511         80875.0     2025-06-10        FY26 DHRM_PUBLIC
        3       SW           33828         93557.0     2025-06-10        FY26 DHRM_PUBLIC
        4       SW           44192        117360.0     2025-06-10        FY26 DHRM_PUBLIC
        5       SW           57733        148455.0     2025-06-10        FY26 DHRM_PUBLIC
        6       SW           75423        189075.0     2025-06-10        FY26 DHRM_PUBLIC
        7       SW           98535        242152.0     2025-06-10        FY26 DHRM_PUBLIC
        8       SW          128721        311485.0     2025-06-10        FY26 DHRM_PUBLIC
        9       SW          168166             NaN     2025-06-10        FY26 DHRM_PUBLIC


## W&M University Salary Structure

William & Mary maintains its own pay grade system separate from the DHRM state bands.
All employees hired since 2009 are university employees on the W&M structure.

- **Salaried grades (S01-S23):** Annual salary ranges used for classified staff positions.
  The minimum wage floor is $32,240/year ($15.50/hr), so S01-S07 share the same minimum.
- **Hourly grades (H01-H23):** Hourly rate equivalents. H01 through H07 are compressed
  into a single range at the minimum wage floor; H08-H23 have distinct ranges.

Source: [W&M University Salary Structure](https://www.wm.edu/offices/uhr/employees/currentemployees/positionsandpay/compensation/universitysalarystructure/)
Effective June 10, 2025.

In [ ]:
# ---------------------------------------------------------------------------
# W&M University Salary Structure (effective 2025-06-10)
# ---------------------------------------------------------------------------
# Source: W&M Office of University HR
# S-grades: salaried (annual). H-grades: hourly.

wm_salaried = [
    ("S01", 32240, 36989, 41738),
    ("S02", 32240, 39076, 45911),
    ("S03", 32240, 41371, 50503),
    ("S04", 32240, 43897, 55553),
    ("S05", 32240, 45784, 59328),
    ("S06", 32240, 49730, 67220),
    ("S07", 32240, 53091, 73942),
    ("S08", 35136, 58235, 81335),
    ("S09", 38649, 64059, 89469),
    ("S10", 42514, 70465, 98417),
    ("S11", 46766, 77512, 108258),
    ("S12", 51442, 85263, 119083),
    ("S13", 56587, 93789, 130991),
    ("S14", 62245, 103168, 144091),
    ("S15", 68470, 113485, 158501),
    ("S16", 75317, 124833, 174349),
    ("S17", 82848, 137317, 191786),
    ("S18", 91133, 151048, 210964),
    ("S19", 100247, 166154, 232060),
    ("S20", 110271, 182769, 255267),
    ("S21", 121299, 201046, 280793),
    ("S22", 133428, 221150, 308871),
    ("S23", 146771, 238318, 339760),
]

wm_hourly = [
    ("H01", 15.50, 17.78, 20.06),
    ("H02", 15.50, 18.68, 21.86),
    ("H03", 15.50, 19.65, 23.79),
    ("H04", 15.50, 20.68, 25.86),
    ("H05", 15.50, 21.79, 28.07),
    ("H06", 15.50, 23.32, 31.14),
    ("H07", 15.50, 25.01, 35.55),
    ("H08", 16.89, 27.99, 39.10),
    ("H09", 18.58, 30.80, 43.01),
    ("H10", 20.44, 33.88, 47.32),
    ("H11", 22.48, 37.26, 52.05),
    ("H12", 24.73, 40.99, 57.25),
    ("H13", 27.21, 45.09, 62.97),
    ("H14", 29.93, 49.60, 69.28),
    ("H15", 32.92, 54.56, 76.20),
    ("H16", 36.21, 60.02, 83.82),
    ("H17", 39.83, 66.02, 92.21),
    ("H18", 43.81, 72.62, 101.42),
    ("H19", 48.20, 79.88, 111.57),
    ("H20", 53.01, 87.87, 122.72),
    ("H21", 58.32, 96.66, 134.99),
    ("H22", 64.15, 106.32, 148.50),
    ("H23", 70.56, 116.95, 163.35),
]

df_wm_salaried = pd.DataFrame(wm_salaried,
    columns=["pay_grade", "annual_min", "annual_midpoint", "annual_max"])
df_wm_salaried["grade_type"] = "salaried"
df_wm_salaried["effective_date"] = "2025-06-10"
df_wm_salaried["data_source"] = "WM_PUBLIC"

df_wm_hourly = pd.DataFrame(wm_hourly,
    columns=["pay_grade", "hourly_min", "hourly_midpoint", "hourly_max"])
df_wm_hourly["grade_type"] = "hourly"
df_wm_hourly["effective_date"] = "2025-06-10"
df_wm_hourly["data_source"] = "WM_PUBLIC"

# Annual equivalents for hourly grades (2,080 hours/year)
df_wm_hourly["annual_min"] = (df_wm_hourly["hourly_min"] * 2080).round().astype(int)
df_wm_hourly["annual_midpoint"] = (df_wm_hourly["hourly_midpoint"] * 2080).round().astype(int)
df_wm_hourly["annual_max"] = (df_wm_hourly["hourly_max"] * 2080).round().astype(int)

df_wm_pay_grades = pd.concat([
    df_wm_salaried[["pay_grade", "grade_type", "annual_min", "annual_midpoint",
                     "annual_max", "effective_date", "data_source"]],
    df_wm_hourly[["pay_grade", "grade_type", "annual_min", "annual_midpoint",
                   "annual_max", "effective_date", "data_source"]],
], ignore_index=True)

print(f"W&M Pay Grades: {len(df_wm_pay_grades)} total "
      f"({len(df_wm_salaried)} salaried, {len(df_wm_hourly)} hourly)")
print(f"\nSalaried grades (S01-S23):")
print(df_wm_salaried[["pay_grade", "annual_min", "annual_midpoint", "annual_max"]]
      .to_string(index=False))

## DHRM-to-W&M Pay Band Crosswalk

Each DHRM pay band (1-9) maps to one or more W&M salaried pay grades (S01-S23) based
on salary range overlap. The crosswalk finds the W&M S-grade whose midpoint is closest
to each DHRM band's midpoint, providing HR with a direct translation between state and
university pay structures.

**Validated example:** DHRM Pay Band 7 ($98,535-$242,152, midpoint ~$170K) maps to
W&M S18 ($91,133-$210,964, midpoint $151K) -- confirmed by the Assistant Controller
classification in the project spec.

In [ ]:
# ---------------------------------------------------------------------------
# DHRM-to-W&M Crosswalk (range-containment matching)
# ---------------------------------------------------------------------------
# For each DHRM pay band, find the highest W&M salaried grade whose
# salary range contains the DHRM band minimum. This represents the
# W&M grade an employee entering at band minimum would be placed in.
# Validated: Band 7 -> S18 (per Assistant Controller example in spec).

crosswalk_rows = []

for _, dhrm_row in df_dhrm_pay_bands.iterrows():
    band = dhrm_row["pay_band"]
    dhrm_min = dhrm_row["minimum_salary"]
    dhrm_max = dhrm_row["maximum_salary"]

    # Find all W&M grades whose range contains the DHRM band minimum
    candidates = df_wm_salaried[
        (df_wm_salaried["annual_min"] <= dhrm_min) &
        (df_wm_salaried["annual_max"] >= dhrm_min)
    ]

    if len(candidates) > 0:
        # Pick the highest grade (last row when sorted by min)
        best = candidates.iloc[-1]
    else:
        # Fallback: closest grade by minimum salary
        salaried = df_wm_salaried.copy()
        salaried["distance"] = abs(salaried["annual_min"] - dhrm_min)
        best = salaried.loc[salaried["distance"].idxmin()]

    crosswalk_rows.append({
        "dhrm_pay_band": band,
        "dhrm_min": int(dhrm_min),
        "dhrm_max": int(dhrm_max) if not pd.isna(dhrm_max) else None,
        "wm_pay_grade": best["pay_grade"],
        "wm_min": best["annual_min"],
        "wm_midpoint": best["annual_midpoint"],
        "wm_max": best["annual_max"],
        "data_source": "DERIVED",
    })

df_crosswalk = pd.DataFrame(crosswalk_rows)
print(f"DHRM-to-W&M Crosswalk ({len(df_crosswalk)} bands):")
print(df_crosswalk[["dhrm_pay_band", "dhrm_min", "dhrm_max",
                     "wm_pay_grade", "wm_min", "wm_midpoint", "wm_max"]]
      .to_string(index=False))

## Stage 6 - Database Load

The pipeline targets a local MySQL instance as the primary data store and falls back to CSV export when the database is unavailable. This two-path design keeps the notebook runnable on any machine without requiring a running database server.

**Load order matters.** The `career_groups` table is loaded first because `roles`, `soc_codes`, and `historical_titles` reference it through their `career_group_code` foreign key. Loading in the wrong order would violate referential integrity constraints.

**Connection credentials** are stored as placeholder variables at the top of the cell. In a production deployment these would be read from environment variables or a secrets manager, never hard-coded.

**CSV fallback** writes all four DataFrames to an `output_csv/` directory so downstream analysis can proceed even without a database. The CSV path also serves as a portable export for colleagues who do not have database access.

In [14]:
# ---------------------------------------------------------------------------
# Cell 18 - Database load with CSV fallback
# ---------------------------------------------------------------------------
from sqlalchemy import create_engine
import os

# Database credentials (replace with real values or use env vars in production)
DB_USER = os.getenv("GRIFFIN_DB_USER", "griffin_user")
DB_PASS = os.getenv("GRIFFIN_DB_PASS", "CHANGE_ME")
DB_HOST = os.getenv("GRIFFIN_DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("GRIFFIN_DB_PORT", "3306")
DB_NAME = os.getenv("GRIFFIN_DB_NAME", "griffin_dhrm")

CONNECTION_STRING = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

db_loaded = False

try:
    engine = create_engine(CONNECTION_STRING, connect_args={"connect_timeout": 5})
    # Test the connection
    with engine.connect() as conn:
        conn.execute("SELECT 1")
    print("Connected to MySQL database successfully.\n")

    # Load in FK-safe order: parent table first
    df_career_groups.to_sql('career_groups', engine, if_exists='append', index=False)
    print(f"  Loaded {len(df_career_groups)} career groups")

    df_roles.to_sql('roles', engine, if_exists='append', index=False)
    print(f"  Loaded {len(df_roles)} roles")

    df_soc_codes.to_sql('soc_codes', engine, if_exists='append', index=False)
    print(f"  Loaded {len(df_soc_codes)} SOC codes")

    df_historical_titles.to_sql('historical_titles', engine, if_exists='append', index=False)
    print(f"  Loaded {len(df_historical_titles)} historical titles")

    df_dhrm_pay_bands.to_sql('dhrm_pay_bands', engine, if_exists='append', index=False)
    print(f"  Loaded {len(df_dhrm_pay_bands)} DHRM pay bands")

    df_wm_pay_grades.to_sql('wm_pay_grades', engine, if_exists='append', index=False)
    print(f"  Loaded {len(df_wm_pay_grades)} W&M pay grades")

    df_crosswalk.to_sql('pay_band_crosswalk', engine, if_exists='append', index=False)
    print(f"  Loaded {len(df_crosswalk)} crosswalk mappings")

    db_loaded = True
    print("\nDatabase load complete.")

except Exception as e:
    print(f"Database connection failed: {e}")
    print("Falling back to CSV export...\n")

    csv_dir = os.path.join(os.getcwd(), "..", "data", "reference")
    os.makedirs(csv_dir, exist_ok=True)

    df_career_groups.to_csv(os.path.join(csv_dir, "career_groups.csv"), index=False)
    print(f"  Exported career_groups.csv ({len(df_career_groups)} rows)")

    df_roles.to_csv(os.path.join(csv_dir, "roles.csv"), index=False)
    print(f"  Exported roles.csv ({len(df_roles)} rows)")

    df_soc_codes.to_csv(os.path.join(csv_dir, "soc_codes.csv"), index=False)
    print(f"  Exported soc_codes.csv ({len(df_soc_codes)} rows)")

    df_historical_titles.to_csv(os.path.join(csv_dir, "historical_titles.csv"), index=False)
    print(f"  Exported historical_titles.csv ({len(df_historical_titles)} rows)")

    df_dhrm_pay_bands.to_csv(os.path.join(csv_dir, "dhrm_pay_bands.csv"), index=False)
    print(f"  Exported dhrm_pay_bands.csv ({len(df_dhrm_pay_bands)} rows)")

    print(f"\nCSV fallback complete. Files saved to: {csv_dir}")

Database connection failed: (pymysql.err.OperationalError) (1045, "Access denied for user 'griffin_user'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Falling back to CSV export...

  Exported career_groups.csv (56 rows)
  Exported roles.csv (294 rows)
  Exported soc_codes.csv (666 rows)
  Exported historical_titles.csv (1469 rows)
  Exported dhrm_pay_bands.csv (9 rows)

CSV fallback complete. Files saved to: C:\Users\salva\Desktop\MSBA\Claude Skills\Courses\AI Course\HR_Classification_Project\output_csv


## HR Handoff Export (Excel)

The project spec requires Excel files that HR can open, filter, and update without
technical skills. This cell exports all reference tables as .xlsx files in a single
handoff folder, matching the deliverable structure from the project specification.

In [ ]:
# ---------------------------------------------------------------------------
# Excel export for HR handoff package
# ---------------------------------------------------------------------------
import openpyxl  # pandas uses this backend for .xlsx

xlsx_dir = os.path.join(os.getcwd(), "..", "data", "reference")
os.makedirs(xlsx_dir, exist_ok=True)

exports = [
    (df_career_groups,    "career_groups.xlsx",    "career_groups"),
    (df_roles,            "roles.xlsx",            "roles"),
    (df_soc_codes,        "soc_codes.xlsx",        "soc_codes"),
    (df_historical_titles, "historical_titles.xlsx", "historical_titles"),
    (df_dhrm_pay_bands,   "dhrm_pay_bands.xlsx",   "dhrm_pay_bands"),
    (df_wm_pay_grades,    "wm_pay_grades.xlsx",    "wm_pay_grades"),
    (df_crosswalk,        "crosswalk.xlsx",        "crosswalk"),
]

print("HR Handoff Excel Export")
print("=" * 50)
for df, filename, label in exports:
    path = os.path.join(xlsx_dir, filename)
    df.to_excel(path, index=False, sheet_name=label)
    print(f"  {filename:<30s} {len(df):>5} rows")

print()
print(f"Files saved to: {xlsx_dir}")


## Stage 7 - Verification Queries

The final stage runs a standard set of summary queries against the loaded data to confirm correctness. When the database connection succeeded, queries run as SQL against MySQL. When the CSV fallback was used, the same statistics are computed directly from the in-memory DataFrames.

Five verification checks are performed:

1. **Record counts** for all four tables.
2. **Career groups by occupational family** to confirm the family-level grouping.
3. **Roles per career group** to spot any groups that parsed with zero roles.
4. **Track distribution** (R = Role, S = Supervisory) across all roles.
5. **Sample role detail** showing one complete role record with all parsed fields.

In [15]:
# ---------------------------------------------------------------------------
# Cell 20 - Verification queries
# ---------------------------------------------------------------------------

if db_loaded:
    # ---- SQL path ----
    print("Verification via SQL queries\n")
    with engine.connect() as conn:
        # 1. Record counts
        print("1. Record Counts")
        for table in ['career_groups', 'roles', 'soc_codes', 'historical_titles']:
            result = conn.execute(f"SELECT COUNT(*) FROM {table}")
            count = result.fetchone()[0]
            print(f"   {table}: {count}")

        # 2. Career groups by occupational family
        print("\n2. Career Groups by Occupational Family")
        df_q2 = pd.read_sql(
            "SELECT occupational_family, COUNT(*) AS group_count "
            "FROM career_groups GROUP BY occupational_family ORDER BY group_count DESC",
            conn
        )
        print(df_q2.to_string(index=False))

        # 3. Roles per career group
        print("\n3. Roles per Career Group")
        df_q3 = pd.read_sql(
            "SELECT r.career_group_code, cg.career_group_name, COUNT(*) AS role_count "
            "FROM roles r JOIN career_groups cg ON r.career_group_code = cg.career_group_code "
            "GROUP BY r.career_group_code, cg.career_group_name ORDER BY role_count DESC",
            conn
        )
        print(df_q3.to_string(index=False))

        # 4. Track distribution
        print("\n4. Track Distribution")
        df_q4 = pd.read_sql(
            "SELECT track, COUNT(*) AS role_count FROM roles GROUP BY track",
            conn
        )
        print(df_q4.to_string(index=False))

        # 5. Sample role detail
        print("\n5. Sample Role Detail")
        df_q5 = pd.read_sql("SELECT * FROM roles LIMIT 1", conn)
        for col in df_q5.columns:
            val = str(df_q5[col].iloc[0])[:100]
            print(f"   {col}: {val}")

else:
    # ---- DataFrame path (CSV fallback) ----
    print("Verification via DataFrames (CSV fallback mode)\n")

    # 1. Record counts
    print("1. Record Counts")
    print(f"   career_groups:    {len(df_career_groups)}")
    print(f"   roles:            {len(df_roles)}")
    print(f"   soc_codes:        {len(df_soc_codes)}")
    print(f"   historical_titles: {len(df_historical_titles)}")

    # 2. Career groups by occupational family
    print("\n2. Career Groups by Occupational Family")
    family_counts = (df_career_groups.groupby('occupational_family')
                     .size().reset_index(name='group_count')
                     .sort_values('group_count', ascending=False))
    print(family_counts.to_string(index=False))

    # 3. Roles per career group
    print("\n3. Roles per Career Group")
    roles_per_group = (df_roles.groupby('career_group_code')
                       .size().reset_index(name='role_count'))
    roles_per_group = roles_per_group.merge(
        df_career_groups[['career_group_code', 'career_group_name']],
        on='career_group_code', how='left'
    ).sort_values('role_count', ascending=False)
    print(roles_per_group[['career_group_code', 'career_group_name', 'role_count']]
          .to_string(index=False))

    # 4. Track distribution
    print("\n4. Track Distribution")
    track_dist = df_roles['track'].value_counts().reset_index()
    track_dist.columns = ['track', 'role_count']
    print(track_dist.to_string(index=False))

    # 5. Sample role detail
    print("\n5. Sample Role Detail")
    sample = df_roles.iloc[0]
    for col in df_roles.columns:
        val = str(sample[col])[:100]
        print(f"   {col}: {val}")

print(f"\n{'='*50}")
print("Pipeline complete.")

Verification via DataFrames (CSV fallback mode)

1. Record Counts
   career_groups:    56
   roles:            294
   soc_codes:        666
   historical_titles: 1469

2. Career Groups by Occupational Family
                  occupational_family  group_count
                Trades and Operations           11
            Health and Human Services           10
              Administrative Services           10
Natural Resources and Applied Science            7
       Educational and Media Services            7
                        Public Safety            6
           Engineering and Technology            5

3. Roles per Career Group
career_group_code                                   career_group_name  role_count
            29090                      Public Relations and Marketing           9
            69110                                   Security Services           9
            59010                               Agricultural Services           8
            19130            